In [109]:
import pandas as pd
import mysql.connector

print("Pandas:", pd.__version__)
print("MySQL Connector: Working")

Pandas: 3.0.5
MySQL Connector: Working


In [110]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.db_connection import get_connection

connection = get_connection()

print("DATABASE CONNECTION SUCCESSFUL")

DATABASE CONNECTION SUCCESSFUL


In [111]:
cursor = connection.cursor()

cursor.execute("SHOW TABLES")

tables = [table[0] for table in cursor.fetchall()]

print("Tables in database:")
for table in tables:
    print("-", table)

Tables in database:
- downtime
- employees
- inventory
- machines
- maintenance
- production
- production_logs
- production_targets
- quality
- sensors
- shifts
- suppliers


In [112]:
data = {}

for table in tables:
    query = f"SELECT * FROM `{table}`"
    data[table] = pd.read_sql(query, connection)

    print(f"{table}: {data[table].shape[0]} rows × {data[table].shape[1]} columns")

downtime: 32 rows × 6 columns
employees: 5 rows × 5 columns
inventory: 5 rows × 6 columns
machines: 5 rows × 6 columns
maintenance: 13 rows × 7 columns
production: 450 rows × 7 columns
production_logs: 450 rows × 6 columns
production_targets: 450 rows × 4 columns
quality: 450 rows × 6 columns
sensors: 900 rows × 6 columns
shifts: 3 rows × 5 columns
suppliers: 5 rows × 6 columns


C:\Users\Sai Sanjana S\AppData\Local\Temp\ipykernel_74920\4262546910.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data[table] = pd.read_sql(query, connection)


In [113]:
for table, df in data.items():
    print("\n" + "=" * 60)
    print(f"TABLE: {table}")
    print("=" * 60)
    print("Columns:", list(df.columns))
    print("\nData types:")
    print(df.dtypes)


TABLE: downtime
Columns: ['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']

Data types:
downtime_id                 int64
machine_id                  int64
downtime_start     datetime64[us]
downtime_end       datetime64[us]
downtime_reason               str
downtime_hours            float64
dtype: object

TABLE: employees
Columns: ['employee_id', 'employee_name', 'department', 'role', 'shift']

Data types:
employee_id      int64
employee_name      str
department         str
role               str
shift              str
dtype: object

TABLE: inventory
Columns: ['inventory_id', 'material_name', 'quantity_available', 'reorder_level', 'unit', 'last_updated']

Data types:
inventory_id           int64
material_name            str
quantity_available     int64
reorder_level          int64
unit                     str
last_updated          object
dtype: object

TABLE: machines
Columns: ['machine_id', 'machine_name', 'machine_type', 'location',

In [114]:
summary = []

for table, df in data.items():
    summary.append({
        "table": table,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isnull().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)

summary_df

,table,rows,columns,missing_values,duplicate_rows
0,downtime,32,6,0,0
1,employees,5,5,0,0
2,inventory,5,6,0,0
3,machines,5,6,0,0
4,maintenance,13,7,0,0
5,production,450,7,0,0
6,production_logs,450,6,0,0
7,production_targets,450,4,0,0
8,quality,450,6,0,0
9,sensors,900,6,0,0


In [115]:
important_tables = [
    "machines",
    "production",
    "production_targets",
    "quality",
    "maintenance",
    "downtime",
    "sensors"
]

for table in important_tables:
    print("\n" + "=" * 70)
    print(f"TABLE: {table}")
    print("=" * 70)
    display(data[table])


TABLE: machines


,machine_id,machine_name,machine_type,location,status,installation_date
0,1,PCB Assembly Line 01,Assembly Line,Production Floor A,Operational,2022-05-10
1,2,PCB Assembly Line 02,Assembly Line,Production Floor A,Operational,2022-08-15
2,3,CNC Precision Unit,CNC Machine,Production Floor B,Maintenance,2021-03-20
3,4,Testing Station 01,Testing Equipment,Quality Floor,Operational,2023-01-12
4,5,Testing Station 02,Testing Equipment,Quality Floor,Operational,2023-04-18



TABLE: production


,production_id,machine_id,production_date,shift,units_produced,units_rejected,production_time_hours
0,1,1,2026-07-20,Morning,969,59,8.0
1,2,1,2026-07-20,Afternoon,944,48,8.0
2,3,1,2026-07-20,Night,940,13,8.0
3,4,2,2026-07-20,Morning,890,62,8.0
4,5,2,2026-07-20,Afternoon,963,11,8.0
...,...,...,...,...,...,...,...
445,446,4,2026-08-18,Afternoon,1050,42,8.0
446,447,4,2026-08-18,Night,1008,66,8.0
447,448,5,2026-08-18,Morning,907,53,8.0
448,449,5,2026-08-18,Afternoon,954,33,8.0



TABLE: production_targets


...

,maintenance_id,equipment_id,maintenance_date,maintenance_type,maintenance_status,downtime_hours,maintenance_cost
0,1,5,2026-07-25,Preventive,Completed,1.03,4534.96
1,2,2,2026-07-27,Corrective,Completed,5.40,8731.18
2,3,3,2026-07-27,Preventive,Completed,1.18,4547.77
3,4,2,2026-07-28,Routine Inspection,Completed,1.30,1758.16
4,5,4,2026-07-29,Routine Inspection,Completed,0.86,1860.77
5,6,5,2026-07-29,Routine Inspection,Completed,1.19,2066.30
6,7,1,2026-07-31,Corrective,Completed,5.38,9108.13
7,8,5,2026-08-03,Routine Inspection,In Progress,1.11,1903.05
8,9,4,2026-08-05,Routine Inspection,Completed,1.47,2078.70
9,10,1,2026-08-08,Preventive,In Progress,1.36,3048.84



TABLE: downtime


,downtime_id,machine_id,downtime_start,downtime_end,downtime_reason,downtime_hours
0,1,3,2026-07-20 06:00:00,2026-07-20 09:28:48,Mechanical Failure,3.48
1,2,2,2026-07-21 16:00:00,2026-07-21 18:08:24,Electrical Issue,2.14
2,3,4,2026-07-21 20:00:00,2026-07-21 21:01:12,Mechanical Failure,1.02
3,4,1,2026-07-22 13:00:00,2026-07-22 15:02:24,Mechanical Failure,2.04
4,5,3,2026-07-22 19:00:00,2026-07-22 19:34:12,Testing Error,0.57
5,6,1,2026-07-23 10:00:00,2026-07-23 11:06:00,Testing Error,1.10
6,7,2,2026-07-23 21:00:00,2026-07-23 21:46:12,Electrical Issue,0.77
7,8,2,2026-07-24 11:00:00,2026-07-24 14:13:48,Electrical Issue,3.23
8,9,3,2026-07-24 17:00:00,2026-07-24 18:27:36,Mechanical Failure,1.46
9,10,4,2026-07-24 12:00:00,2026-07-24 15:56:24,Machine Adjustment,3.94



TABLE: sensors


,sensor_id,machine_id,sensor_type,sensor_value,unit,recorded_at
0,1,1,Temperature,71.73,°C,2026-07-20 08:00:00
1,2,1,Vibration,1.80,mm/s,2026-07-20 08:00:00
2,3,1,Temperature,66.20,°C,2026-07-20 14:00:00
3,4,1,Vibration,3.02,mm/s,2026-07-20 14:00:00
4,5,1,Temperature,68.79,°C,2026-07-20 22:00:00
...,...,...,...,...,...,...
895,896,5,Vibration,3.50,mm/s,2026-08-18 08:00:00
896,897,5,Temperature,68.75,°C,2026-08-18 14:00:00
897,898,5,Vibration,2.78,mm/s,2026-08-18 14:00:00
898,899,5,Temperature,65.67,°C,2026-08-18 22:00:00


In [116]:
for table in important_tables:
    print("\n" + "=" * 60)
    print(f"NUMERIC SUMMARY: {table}")
    print("=" * 60)
    display(data[table].describe())


NUMERIC SUMMARY: machines


,machine_id
count,5.000000
mean,3.000000
std,1.581139
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,5.000000



NUMERIC SUMMARY: production


,production_id,machine_id,units_produced,units_rejected,production_time_hours
count,450.000000,450.000000,450.000000,450.000000,450.0
mean,225.500000,3.000000,911.237778,40.791111,8.0
std,130.048068,1.415788,73.708261,19.310057,0.0
min,1.000000,1.000000,670.000000,8.000000,8.0
25%,113.250000,2.000000,868.000000,24.000000,8.0
50%,225.500000,3.000000,921.000000,42.000000,8.0
75%,337.750000,4.000000,961.000000,58.750000,8.0
max,450.000000,5.000000,1134.000000,83.000000,8.0



NUMERIC SUMMARY: production_targets


...

,maintenance_id,equipment_id,downtime_hours,maintenance_cost
count,13.00000,13.000000,13.000000,13.000000
mean,7.00000,3.307692,2.122308,4771.213077
std,3.89444,1.493576,1.693636,3217.972525
min,1.00000,1.000000,0.860000,1758.160000
25%,4.00000,2.000000,1.110000,2066.300000
50%,7.00000,3.000000,1.300000,4534.960000
75%,10.00000,5.000000,2.040000,5924.240000
max,13.00000,5.000000,5.400000,11478.270000



NUMERIC SUMMARY: downtime


,downtime_id,machine_id,downtime_start,downtime_end,downtime_hours
count,32.000000,32.000000,32,32,32.000000
mean,16.500000,2.937500,2026-08-02 20:33:45,2026-08-02 22:46:36.750000,2.214375
min,1.000000,1.000000,2026-07-20 06:00:00,2026-07-20 09:28:48,0.570000
25%,8.750000,2.000000,2026-07-24 11:45:00,2026-07-24 15:30:45,1.335000
50%,16.500000,3.000000,2026-08-01 08:00:00,2026-08-01 11:15:18,2.150000
75%,24.250000,4.000000,2026-08-10 22:30:00,2026-08-10 23:32:06,3.177500
max,32.000000,5.000000,2026-08-18 13:00:00,2026-08-18 16:21:36,3.940000
std,9.380832,1.134147,NaN,NaN,1.047661



NUMERIC SUMMARY: sensors


,sensor_id,machine_id,sensor_value,recorded_at
count,900.000000,900.000,900.000000,900
mean,450.500000,3.000,36.894000,2026-08-04 02:40:00
min,1.000000,1.000,1.240000,2026-07-20 08:00:00
25%,225.750000,2.000,2.897500,2026-07-27 14:00:00
50%,450.500000,3.000,31.640000,2026-08-04 03:00:00
75%,675.250000,4.000,68.722500,2026-08-11 14:00:00
max,900.000000,5.000,88.870000,2026-08-18 22:00:00
std,259.951919,1.415,34.034031,NaN


In [117]:
for table in important_tables:
    print("\n" + "=" * 60)
    print(f"CATEGORICAL VALUES: {table}")
    print("=" * 60)

    for column in data[table].select_dtypes(include=["object", "str"]).columns:
        print(f"\n{column}:")
        print(data[table][column].unique())


CATEGORICAL VALUES: machines

machine_name:
<StringArray>
['PCB Assembly Line 01', 'PCB Assembly Line 02',   'CNC Precision Unit',
   'Testing Station 01',   'Testing Station 02']
Length: 5, dtype: str

machine_type:
<StringArray>
['Assembly Line', 'CNC Machine', 'Testing Equipment']
Length: 3, dtype: str

location:
<StringArray>
['Production Floor A', 'Production Floor B', 'Quality Floor']
Length: 3, dtype: str

status:
<StringArray>
['Operational', 'Maintenance']
Length: 2, dtype: str

installation_date:
[datetime.date(2022, 5, 10) datetime.date(2022, 8, 15)
 datetime.date(2021, 3, 20) datetime.date(2023, 1, 12)
 datetime.date(2023, 4, 18)]

CATEGORICAL VALUES: production

production_date:
[datetime.date(2026, 7, 20) datetime.date(2026, 7, 21)
 datetime.date(2026, 7, 22) datetime.date(2026, 7, 23)
 datetime.date(2026, 7, 24) datetime.date(2026, 7, 25)
 datetime.date(2026, 7, 26) datetime.date(2026, 7, 27)
 datetime.date(2026, 7, 28) datetime.date(2026, 7, 29)
 datetime.date(2026, 7,

In [118]:
print("Machines referenced in production:")
print(sorted(data["production"]["machine_id"].unique()))

print("\nMachines in machines table:")
print(sorted(data["machines"]["machine_id"].unique()))

print("\nProduction IDs referenced in quality:")
print(sorted(data["quality"]["production_id"].unique()))

print("\nProduction IDs in production table:")
print(sorted(data["production"]["production_id"].unique()))

print("\nMachines referenced in downtime:")
print(sorted(data["downtime"]["machine_id"].unique()))

print("\nMachines referenced in sensors:")
print(sorted(data["sensors"]["machine_id"].unique()))

Machines referenced in production:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Machines in machines table:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Production IDs referenced in quality:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.

In [119]:
# ============================================================
# DATA CLEANING PIPELINE
# CHECK FOR  MISSING VALUES
# ============================================================

print("MISSING VALUE CHECK")
print("=" * 70)

for table, df in data.items():
    print(f"\nTABLE: {table}")
    print("-" * 50)

    missing = df.isnull().sum()

    if missing.sum() == 0:
        print("No missing values found.")
    else:
        print(missing[missing > 0])

MISSING VALUE CHECK

TABLE: downtime
--------------------------------------------------
No missing values found.

TABLE: employees
--------------------------------------------------
No missing values found.

TABLE: inventory
--------------------------------------------------
No missing values found.

TABLE: machines
--------------------------------------------------
No missing values found.

TABLE: maintenance
--------------------------------------------------
No missing values found.

TABLE: production
--------------------------------------------------
No missing values found.

TABLE: production_logs
--------------------------------------------------
No missing values found.

TABLE: production_targets
--------------------------------------------------
No missing values found.

TABLE: quality
--------------------------------------------------
No missing values found.

TABLE: sensors
--------------------------------------------------
No missing values found.

TABLE: shifts
-------------

In [120]:
# ============================================================
# CHECK DUPLICATE ROWS
# ============================================================

print("DUPLICATE ROW CHECK")
print("=" * 70)

for table, df in data.items():
    duplicate_count = df.duplicated().sum()

    print(f"{table}: {duplicate_count} duplicate rows")

DUPLICATE ROW CHECK
downtime: 0 duplicate rows
employees: 0 duplicate rows
inventory: 0 duplicate rows
machines: 0 duplicate rows
maintenance: 0 duplicate rows
production: 0 duplicate rows
production_logs: 0 duplicate rows
production_targets: 0 duplicate rows
quality: 0 duplicate rows
sensors: 0 duplicate rows
shifts: 0 duplicate rows
suppliers: 0 duplicate rows


In [121]:




# CHECK INVALID NUMERIC VALUES
# ============================================================

print("INVALID NUMERIC VALUE CHECK")
print("=" * 70)

for table, df in data.items():

    print(f"\nTABLE: {table}")
    print("-" * 50)

    found_invalid = False

    for col in df.columns:

        # Skip datetime columns
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue

        # Skip timedelta columns
        if pd.api.types.is_timedelta64_dtype(df[col]):
            continue

        # Check only actual numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            negative_count = (df[col] < 0).sum()

            if negative_count > 0:
                print(f"{col}: {negative_count} negative values")
                found_invalid = True

    if not found_invalid:
        print("No negative numeric values found.")

INVALID NUMERIC VALUE CHECK

TABLE: downtime
--------------------------------------------------
No negative numeric values found.

TABLE: employees
--------------------------------------------------
No negative numeric values found.

TABLE: inventory
--------------------------------------------------
No negative numeric values found.

TABLE: machines
--------------------------------------------------
No negative numeric values found.

TABLE: maintenance
--------------------------------------------------
No negative numeric values found.

TABLE: production
--------------------------------------------------
No negative numeric values found.

TABLE: production_logs
--------------------------------------------------
No negative numeric values found.

TABLE: production_targets
--------------------------------------------------
No negative numeric values found.

TABLE: quality
--------------------------------------------------
No negative numeric values found.

TABLE: sensors
---------------

In [122]:

# DATE/TIME VALIDATION


print("DATE/TIME VALIDATION")
print("=" * 70)

for table, df in data.items():

    datetime_cols = df.select_dtypes(
        include=["datetime", "datetimetz"]
    ).columns

    print(f"\nTABLE: {table}")
    print("-" * 50)

    if len(datetime_cols) == 0:
        print("No datetime columns found.")
        continue

    for col in datetime_cols:
        print(f"{col}:")
        print(f"  Minimum: {df[col].min()}")
        print(f"  Maximum: {df[col].max()}")
        print(f"  Missing: {df[col].isna().sum()}")

DATE/TIME VALIDATION

TABLE: downtime
--------------------------------------------------
downtime_start:
  Minimum: 2026-07-20 06:00:00
  Maximum: 2026-08-18 13:00:00
  Missing: 0
downtime_end:
  Minimum: 2026-07-20 09:28:48
  Maximum: 2026-08-18 16:21:36
  Missing: 0

TABLE: employees
--------------------------------------------------
No datetime columns found.

TABLE: inventory
--------------------------------------------------
No datetime columns found.

TABLE: machines
--------------------------------------------------
No datetime columns found.

TABLE: maintenance
--------------------------------------------------
No datetime columns found.

TABLE: production
--------------------------------------------------
No datetime columns found.

TABLE: production_logs
--------------------------------------------------
log_date:
  Minimum: 2026-07-20 00:00:00
  Maximum: 2026-08-18 00:00:00
  Missing: 0

TABLE: production_targets
--------------------------------------------------
No datetime

In [123]:
print("BUSINESS RULE VALIDATION")
print("=" * 70)

# Production checks
production = data["production"]

print("\nPRODUCTION")
print("-" * 50)

if "units_produced" in production.columns:
    print("Negative production:",
          (production["units_produced"] < 0).sum())

if "target_quantity" in production.columns:
    print("Negative target quantity:",
          (production["target_quantity"] < 0).sum())

# Quality checks
quality = data["quality"]

print("\nQUALITY")
print("-" * 50)

if "defect_quantity" in quality.columns:
    print("Negative defects:",
          (quality["defect_quantity"] < 0).sum())

# Downtime checks
downtime = data["downtime"]

print("\nDOWNTIME")
print("-" * 50)

if "downtime_start" in downtime.columns and "downtime_end" in downtime.columns:
    invalid_downtime = (
        downtime["downtime_end"] < downtime["downtime_start"]
    ).sum()

    print("Invalid downtime periods:", invalid_downtime)

# Production target checks
targets = data["production_targets"]

print("\nPRODUCTION TARGETS")
print("-" * 50)

numeric_target_cols = targets.select_dtypes(include="number").columns

for col in numeric_target_cols:
    print(f"{col} negative values:", (targets[col] < 0).sum())

BUSINESS RULE VALIDATION

PRODUCTION
--------------------------------------------------
Negative production: 0

QUALITY
--------------------------------------------------

DOWNTIME
--------------------------------------------------
Invalid downtime periods: 0

PRODUCTION TARGETS
--------------------------------------------------
target_id negative values: 0
machine_id negative values: 0
target_quantity negative values: 0


In [124]:
print("DATA TYPE VALIDATION")
print("=" * 70)

for table, df in data.items():

    print(f"\nTABLE: {table}")
    print("-" * 50)

    for column in df.columns:
        print(f"{column}: {df[column].dtype}")

DATA TYPE VALIDATION

TABLE: downtime
--------------------------------------------------
downtime_id: int64
machine_id: int64
downtime_start: datetime64[us]
downtime_end: datetime64[us]
downtime_reason: str
downtime_hours: float64

TABLE: employees
--------------------------------------------------
employee_id: int64
employee_name: str
department: str
role: str
shift: str

TABLE: inventory
--------------------------------------------------
inventory_id: int64
material_name: str
quantity_available: int64
reorder_level: int64
unit: str
last_updated: object

TABLE: machines
--------------------------------------------------
machine_id: int64
machine_name: str
machine_type: str
location: str
status: str
installation_date: object

TABLE: maintenance
--------------------------------------------------
maintenance_id: int64
equipment_id: int64
maintenance_date: object
maintenance_type: str
maintenance_status: str
downtime_hours: float64
maintenance_cost: float64

TABLE: production
------------

In [125]:
print("OUTLIER CHECK")
print("=" * 70)

for table, df in data.items():

    numeric_cols = df.select_dtypes(include=["number"]).columns

    print(f"\nTABLE: {table}")
    print("-" * 50)

    if len(numeric_cols) == 0:
        print("No numeric columns found.")
        continue

    for col in numeric_cols:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = (
            (df[col] < lower_bound) |
            (df[col] > upper_bound)
        ).sum()

        print(f"{col}: {outliers} potential outliers")

OUTLIER CHECK

TABLE: downtime
--------------------------------------------------
downtime_id: 0 potential outliers
machine_id: 0 potential outliers
downtime_hours: 0 potential outliers

TABLE: employees
--------------------------------------------------
employee_id: 0 potential outliers

TABLE: inventory
--------------------------------------------------
inventory_id: 0 potential outliers
quantity_available: 0 potential outliers
reorder_level: 0 potential outliers

TABLE: machines
--------------------------------------------------
machine_id: 0 potential outliers

TABLE: maintenance
--------------------------------------------------
maintenance_id: 0 potential outliers
equipment_id: 0 potential outliers
downtime_hours: 3 potential outliers
maintenance_cost: 0 potential outliers

TABLE: production
--------------------------------------------------
production_id: 0 potential outliers
machine_id: 0 potential outliers
units_produced: 6 potential outliers


units_rejected: 0 potential outliers
production_time_hours: 0 potential outliers

TABLE: production_logs
--------------------------------------------------
log_id: 0 potential outliers
production_id: 0 potential outliers
machine_id: 0 potential outliers
employee_id: 0 potential outliers
quantity_produced: 6 potential outliers

TABLE: production_targets
--------------------------------------------------
target_id: 0 potential outliers
machine_id: 0 potential outliers
target_quantity: 0 potential outliers

TABLE: quality
--------------------------------------------------
quality_id: 0 potential outliers
production_id: 0 potential outliers
defect_count: 0 potential outliers

TABLE: sensors
--------------------------------------------------
sensor_id: 0 potential outliers
machine_id: 0 potential outliers
sensor_value: 0 potential outliers

TABLE: shifts
--------------------------------------------------
shift_id: 0 potential outliers
start_time: 0 potential outliers
end_time: 0 potential o

In [126]:
 PRODUCTION EDA

production = data["production"]

print("PRODUCTION DATA")
print("=" * 70)

print("\nShape:")
print(production.shape)

print("\nColumns:")
print(production.columns.tolist())

print("\nFirst 5 rows:")
display(production.head())

print("\nStatistical Summary:")
display(production.describe())

IndentationError: unexpected indent (2182687487.py, line 1)

In [ ]:
# PRODUCTION ANALYSIS

print("PRODUCTION ANALYSIS")
print("=" * 70)

# Total production
print("\nTotal Units Produced:")
print(production["units_produced"].sum())

# Average production
print("\nAverage Units Produced:")
print(production["units_produced"].mean())

# Minimum production
print("\nMinimum Units Produced:")
print(production["units_produced"].min())

# Maximum production
print("\nMaximum Units Produced:")
print(production["units_produced"].max())

# Machine-wise production
print("\nMachine-wise Production:")
machine_production = (
    production.groupby("machine_id")["units_produced"]
    .agg(["sum", "mean", "min", "max"])
    .reset_index()
)

display(machine_production)

PRODUCTION ANALYSIS

Total Units Produced:
410057

Average Units Produced:
911.2377777777778

Minimum Units Produced:
670

Maximum Units Produced:
1134

Machine-wise Production:


,machine_id,sum,mean,min,max
0,1,84982,944.244444,842,1034
1,2,81096,901.066667,817,973
2,3,71938,799.311111,670,902
3,4,88769,986.322222,903,1134
4,5,83272,925.244444,845,1022


In [ ]:
# SHIFT-WISE PRODUCTION EDA

print("SHIFT-WISE PRODUCTION ANALYSIS")
print("=" * 70)

shift_production = (
    production.groupby("shift")["units_produced"]
    .agg(["sum", "mean", "min", "max"])
    .reset_index()
)

display(shift_production)

SHIFT-WISE PRODUCTION ANALYSIS


,shift,sum,mean,min,max
0,Afternoon,136822,912.146667,711,1066
1,Morning,136997,913.313333,723,1134
2,Night,136238,908.253333,670,1070


In [ ]:
# TARGET VS ACTUAL PRODUCTION

print("TARGET VS ACTUAL PRODUCTION")
print("=" * 70)

targets = data["production_targets"]

print("\nTarget Data:")
print("Shape:", targets.shape)
print("Columns:", targets.columns.tolist())

display(targets.head())

TARGET VS ACTUAL PRODUCTION

Target Data:
Shape: (450, 4)
Columns: ['target_id', 'machine_id', 'target_date', 'target_quantity']


,target_id,machine_id,target_date,target_quantity
0,1,1,2026-07-20,998
1,2,1,2026-07-20,1012
2,3,1,2026-07-20,1003
3,4,2,2026-07-20,873
4,5,2,2026-07-20,907


In [ ]:
targets = data["production_targets"]

print("TARGET TABLE")
print("=" * 70)
print("Shape:", targets.shape)
print("Columns:", targets.columns.tolist())
print("\nFirst 5 rows:")
print(targets.head().to_string())

TARGET TABLE
Shape: (450, 4)
Columns: ['target_id', 'machine_id', 'target_date', 'target_quantity']

First 5 rows:
   target_id  machine_id target_date  target_quantity
0          1           1  2026-07-20              998
1          2           1  2026-07-20             1012
2          3           1  2026-07-20             1003
3          4           2  2026-07-20              873
4          5           2  2026-07-20              907


In [ ]:
# TARGET VS ACTUAL PRODUCTION

target_vs_actual = production.merge(
    targets,
    left_on=["machine_id", "production_date"],
    right_on=["machine_id", "target_date"],
    how="left"
)

target_vs_actual["achievement_percent"] = (
    target_vs_actual["units_produced"]
    / target_vs_actual["target_quantity"]
) * 100

target_vs_actual["difference"] = (
    target_vs_actual["units_produced"]
    - target_vs_actual["target_quantity"]
)

print("TARGET VS ACTUAL PRODUCTION")
print("=" * 70)

print("\nOverall Target Quantity:")
print(target_vs_actual["target_quantity"].sum())

print("\nOverall Actual Production:")
print(target_vs_actual["units_produced"].sum())

print("\nOverall Achievement %:")
print(
    target_vs_actual["units_produced"].sum()
    / target_vs_actual["target_quantity"].sum()
    * 100
)

print("\nMachine-wise Target vs Actual:")
machine_target_actual = (
    target_vs_actual
    .groupby("machine_id")
    .agg(
        target_quantity=("target_quantity", "sum"),
        actual_production=("units_produced", "sum")
    )
    .reset_index()
)

machine_target_actual["achievement_percent"] = (
    machine_target_actual["actual_production"]
    / machine_target_actual["target_quantity"]
) * 100

machine_target_actual["difference"] = (
    machine_target_actual["actual_production"]
    - machine_target_actual["target_quantity"]
)

print(machine_target_actual.to_string(index=False))

TARGET VS ACTUAL PRODUCTION

Overall Target Quantity:
1258032

Overall Actual Production:
1230171

Overall Achievement %:
97.78535045213476

Machine-wise Target vs Actual:
 machine_id  target_quantity  actual_production  achievement_percent  difference
          1           271278             254946            93.979608      -16332
          2           242712             243288           100.237318         576
          3           216387             215814            99.735197        -573
          4           270837             266307            98.327407       -4530
          5           256818             249816            97.273556       -7002


In [ ]:
# QUALITY / DEFECT EDA

quality = data["quality"]

print("QUALITY DATA")
print("=" * 70)

print("Shape:", quality.shape)
print("Columns:", quality.columns.tolist())

print("\nFirst 5 rows:")
print(quality.head().to_string())

print("\nStatistical Summary:")
print(quality.describe().to_string())

QUALITY DATA
Shape: (450, 6)
Columns: ['quality_id', 'production_id', 'inspection_date', 'defect_type', 'defect_count', 'quality_status']

First 5 rows:
   quality_id  production_id inspection_date             defect_type  defect_count quality_status
0           1              1      2026-07-20              PCB Damage            17         Fail\r
1           2              2      2026-07-20             Overheating             9         Pass\r
2           3              3      2026-07-20           Solder Defect             3         Pass\r
3           4              4      2026-07-20              PCB Damage            24         Fail\r
4           5              5      2026-07-20  Component Misplacement             3         Pass\r

Statistical Summary:
       quality_id  production_id  defect_count
count  450.000000     450.000000    450.000000
mean   225.500000     225.500000     10.522222
std    130.048068     130.048068      6.258051
min      1.000000       1.000000      1.000000
25

In [ ]:
#  QUALITY STATUS ANALYSIS

print("QUALITY STATUS ANALYSIS")
print("=" * 70)

print("\nQuality Status Counts:")
print(quality["quality_status"].value_counts().to_string())

print("\nQuality Status Percentage:")
print(
    (quality["quality_status"].value_counts(normalize=True) * 100)
    .round(2)
    .to_string()
)

QUALITY STATUS ANALYSIS

Quality Status Counts:
quality_status
Pass\r    332
Fail\r    118

Quality Status Percentage:
quality_status
Pass\r    73.78
Fail\r    26.22


In [ ]:
# DEFECT TYPE ANALYSIS

print("DEFECT TYPE ANALYSIS")
print("=" * 70)

defect_analysis = (
    quality.groupby("defect_type")["defect_count"]
    .agg(["count", "sum", "mean", "min", "max"])
    .sort_values("sum", ascending=False)
    .reset_index()
)

print("\nDefect Type Summary:")
print(defect_analysis.to_string(index=False))

DEFECT TYPE ANALYSIS

Defect Type Summary:
           defect_type  count  sum      mean  min  max
     Missing Component    110 1139 10.354545    1   27
         Solder Defect     97 1079 11.123711    2   24
Component Misplacement     74  892 12.054054    2   26
            PCB Damage     85  871 10.247059    1   24
           Overheating     84  754  8.976190    1   26


In [ ]:
# STAGE 17.5.3 — MACHINE-WISE QUALITY ANALYSIS

print("MACHINE-WISE QUALITY ANALYSIS")
print("=" * 70)

# Connect quality data with production data to get machine_id
quality_machine = quality.merge(
    production[["production_id", "machine_id"]],
    on="production_id",
    how="left"
)

machine_quality = (
    quality_machine.groupby("machine_id")
    .agg(
        total_defects=("defect_count", "sum"),
        average_defects=("defect_count", "mean"),
        inspections=("quality_id", "count")
    )
    .reset_index()
)

machine_quality["defects_per_inspection"] = (
    machine_quality["total_defects"]
    / machine_quality["inspections"]
)

print("\nMachine-wise Quality Summary:")
print(machine_quality.to_string(index=False))

MACHINE-WISE QUALITY ANALYSIS

Machine-wise Quality Summary:
 machine_id  total_defects  average_defects  inspections  defects_per_inspection
          1            948        10.533333           90               10.533333
          2           1002        11.133333           90               11.133333
          3            854         9.488889           90                9.488889
          4           1101        12.233333           90               12.233333
          5            830         9.222222           90                9.222222


In [ ]:
#   SENSOR EDA

sensors = data["sensors"]

print("SENSOR DATA")
print("=" * 70)

print("Shape:", sensors.shape)
print("Columns:", sensors.columns.tolist())

print("\nFirst 5 rows:")
print(sensors.head().to_string())

print("\nStatistical Summary:")
print(sensors.describe().to_string())

SENSOR DATA
Shape: (900, 6)
Columns: ['sensor_id', 'machine_id', 'sensor_type', 'sensor_value', 'unit', 'recorded_at']

First 5 rows:
   sensor_id  machine_id  sensor_type  sensor_value  unit         recorded_at
0          1           1  Temperature         71.73    °C 2026-07-20 08:00:00
1          2           1    Vibration          1.80  mm/s 2026-07-20 08:00:00
2          3           1  Temperature         66.20    °C 2026-07-20 14:00:00
3          4           1    Vibration          3.02  mm/s 2026-07-20 14:00:00
4          5           1  Temperature         68.79    °C 2026-07-20 22:00:00

Statistical Summary:
        sensor_id  machine_id  sensor_value          recorded_at
count  900.000000     900.000    900.000000                  900
mean   450.500000       3.000     36.894000  2026-08-04 02:40:00
min      1.000000       1.000      1.240000  2026-07-20 08:00:00
25%    225.750000       2.000      2.897500  2026-07-27 14:00:00
50%    450.500000       3.000     31.640000  2026-0

In [ ]:
#  SENSOR TYPE ANALYSIS

print("SENSOR TYPE ANALYSIS")
print("=" * 70)

sensor_type_summary = (
    sensors.groupby("sensor_type")["sensor_value"]
    .agg(["count", "mean", "min", "max", "std"])
    .reset_index()
)

print("\nSensor Type Summary:")
print(sensor_type_summary.to_string(index=False))

SENSOR TYPE ANALYSIS

Sensor Type Summary:
sensor_type  count      mean   min   max      std
Temperature    450 70.566333 57.23 88.87 6.737070
  Vibration    450  3.221667  1.24  6.05 1.058174


In [ ]:
# MACHINE-WISE SENSOR ANALYSIS

print("MACHINE-WISE SENSOR ANALYSIS")
print("=" * 70)

machine_sensor_summary = (
    sensors.groupby(["machine_id", "sensor_type"])["sensor_value"]
    .agg(["count", "mean", "min", "max", "std"])
    .reset_index()
)

print("\nMachine-wise Sensor Summary:")
print(machine_sensor_summary.to_string(index=False))

MACHINE-WISE SENSOR ANALYSIS

Machine-wise Sensor Summary:
 machine_id sensor_type  count      mean   min   max      std
          1 Temperature     90 69.953222 61.23 76.56 3.333519
          1   Vibration     90  3.005111  1.31  4.22 0.545425
          2 Temperature     90 68.170333 60.45 75.40 3.001107
          2   Vibration     90  2.765222  1.60  3.95 0.472462
          3 Temperature     90 82.204000 73.72 88.87 3.052608
          3   Vibration     90  5.074000  3.76  6.05 0.450674
          4 Temperature     90 65.419111 57.23 75.49 2.914622
          4   Vibration     90  2.547667  1.24  3.78 0.468955
          5 Temperature     90 67.085000 60.50 75.02 2.976255
          5   Vibration     90  2.716333  1.60  3.90 0.509746


In [ ]:
# SENSOR TREND ANALYSIS

print("SENSOR TREND ANALYSIS")
print("=" * 70)

sensor_trends = (
    sensors.groupby(
        [sensors["recorded_at"].dt.date, "sensor_type"]
    )["sensor_value"]
    .agg(["mean", "min", "max"])
    .reset_index()
)

print("\nDaily Sensor Trends:")
print(sensor_trends.to_string(index=False))

SENSOR TREND ANALYSIS

Daily Sensor Trends:
recorded_at sensor_type      mean   min   max
 2026-07-20 Temperature 69.601333 61.66 82.82
 2026-07-20   Vibration  3.079333  1.74  5.60
 2026-07-21 Temperature 69.848667 61.28 85.19
 2026-07-21   Vibration  3.086667  2.25  5.28
 2026-07-22 Temperature 71.417333 62.27 82.98
 2026-07-22   Vibration  3.202667  1.74  5.81
 2026-07-23 Temperature 71.374667 63.40 88.72
 2026-07-23   Vibration  3.170000  1.98  5.23
 2026-07-24 Temperature 71.696667 60.57 83.34
 2026-07-24   Vibration  3.184667  1.99  5.51
 2026-07-25 Temperature 69.204667 60.78 79.39
 2026-07-25   Vibration  3.356667  2.17  5.47
 2026-07-26 Temperature 70.712667 62.01 83.19
 2026-07-26   Vibration  3.042667  2.01  5.08
 2026-07-27 Temperature 71.728667 62.42 85.01
 2026-07-27   Vibration  3.316000  2.30  5.74
 2026-07-28 Temperature 69.956000 63.43 87.65
 2026-07-28   Vibration  3.387333  2.12  5.96
 2026-07-29 Temperature 71.522000 61.93 84.71
 2026-07-29   Vibration  3.175333  1

In [ ]:
# DOWNTIME EDA

downtime = data["downtime"]

print("DOWNTIME DATA")
print("=" * 70)

print("Shape:", downtime.shape)
print("Columns:", downtime.columns.tolist())

print("\nFirst 5 rows:")
print(downtime.head().to_string())

print("\nStatistical Summary:")
print(downtime.describe().to_string())

DOWNTIME DATA
Shape: (32, 6)
Columns: ['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']

First 5 rows:
   downtime_id  machine_id      downtime_start        downtime_end     downtime_reason  downtime_hours
0            1           3 2026-07-20 06:00:00 2026-07-20 09:28:48  Mechanical Failure            3.48
1            2           2 2026-07-21 16:00:00 2026-07-21 18:08:24    Electrical Issue            2.14
2            3           4 2026-07-21 20:00:00 2026-07-21 21:01:12  Mechanical Failure            1.02
3            4           1 2026-07-22 13:00:00 2026-07-22 15:02:24  Mechanical Failure            2.04
4            5           3 2026-07-22 19:00:00 2026-07-22 19:34:12       Testing Error            0.57

Statistical Summary:
       downtime_id  machine_id       downtime_start                downtime_end  downtime_hours
count    32.000000   32.000000                   32                          32       32.000000
mean     16.50

In [ ]:
# DOWNTIME REASON ANALYSIS

print("DOWNTIME REASON ANALYSIS")
print("=" * 70)

downtime_reason = (
    downtime.groupby("downtime_reason")["downtime_hours"]
    .agg(["count", "sum", "mean", "min", "max"])
    .sort_values("sum", ascending=False)
    .reset_index()
)

print("\nDowntime Reason Summary:")
print(downtime_reason.to_string(index=False))

DOWNTIME REASON ANALYSIS

Downtime Reason Summary:
   downtime_reason  count   sum     mean  min  max
Mechanical Failure      8 18.54 2.317500 1.02 3.77
  Electrical Issue      8 15.08 1.885000 0.67 3.23
Machine Adjustment      5 14.21 2.842000 1.60 3.94
       Calibration      3  9.35 3.116667 2.20 3.66
     Testing Error      6  9.13 1.521667 0.57 2.40
 Material Shortage      2  4.55 2.275000 0.94 3.61


In [ ]:
# MACHINE-WISE DOWNTIME ANALYSIS

print("MACHINE-WISE DOWNTIME ANALYSIS")
print("=" * 70)

machine_downtime = (
    downtime.groupby("machine_id")["downtime_hours"]
    .agg(["count", "sum", "mean", "min", "max"])
    .reset_index()
)

machine_downtime.columns = [
    "machine_id",
    "downtime_events",
    "total_downtime_hours",
    "average_downtime_hours",
    "minimum_downtime_hours",
    "maximum_downtime_hours"
]

print("\nMachine-wise Downtime Summary:")
print(machine_downtime.to_string(index=False))

MACHINE-WISE DOWNTIME ANALYSIS

Machine-wise Downtime Summary:
 machine_id  downtime_events  total_downtime_hours  average_downtime_hours  minimum_downtime_hours  maximum_downtime_hours
          1                4                  7.06                1.765000                    1.10                    2.15
          2                6                 15.49                2.581667                    0.77                    3.66
          3               13                 26.39                2.030000                    0.57                    3.77
          4                6                 16.30                2.716667                    1.02                    3.94
          5                3                  5.62                1.873333                    0.94                    3.36


In [ ]:
#  MAINTENANCE EDA

maintenance = data["maintenance"]

print("MAINTENANCE DATA")
print("=" * 70)

print("Shape:", maintenance.shape)
print("Columns:", maintenance.columns.tolist())

print("\nFirst 5 rows:")
print(maintenance.head().to_string())

print("\nStatistical Summary:")
print(maintenance.describe().to_string())

MAINTENANCE DATA
Shape: (13, 7)
Columns: ['maintenance_id', 'equipment_id', 'maintenance_date', 'maintenance_type', 'maintenance_status', 'downtime_hours', 'maintenance_cost']

First 5 rows:
   maintenance_id  equipment_id maintenance_date    maintenance_type maintenance_status  downtime_hours  maintenance_cost
0               1             5       2026-07-25          Preventive          Completed            1.03           4534.96
1               2             2       2026-07-27          Corrective          Completed            5.40           8731.18
2               3             3       2026-07-27          Preventive          Completed            1.18           4547.77
3               4             2       2026-07-28  Routine Inspection          Completed            1.30           1758.16
4               5             4       2026-07-29  Routine Inspection          Completed            0.86           1860.77

Statistical Summary:
       maintenance_id  equipment_id  downtime_hours  ma

In [ ]:
#  MAINTENANCE TYPE ANALYSIS

print("MAINTENANCE TYPE ANALYSIS")
print("=" * 70)

maintenance_type_summary = (
    maintenance.groupby("maintenance_type")
    .agg(
        maintenance_count=("maintenance_id", "count"),
        total_downtime=("downtime_hours", "sum"),
        avg_downtime=("downtime_hours", "mean"),
        total_cost=("maintenance_cost", "sum"),
        avg_cost=("maintenance_cost", "mean")
    )
    .reset_index()
)

print(maintenance_type_summary.to_string(index=False))

MAINTENANCE TYPE ANALYSIS
  maintenance_type  maintenance_count  total_downtime  avg_downtime  total_cost    avg_cost
        Corrective                  3           15.04      5.013333    29317.58 9772.526667
        Preventive                  5            6.62      1.324000    23041.21 4608.242000
Routine Inspection                  5            5.93      1.186000     9666.98 1933.396000


In [ ]:
# MACHINE-WISE MAINTENANCE ANALYSIS

print("MACHINE-WISE MAINTENANCE ANALYSIS")
print("=" * 70)

machine_maintenance = (
    maintenance.groupby("equipment_id")
    .agg(
        maintenance_count=("maintenance_id", "count"),
        total_downtime=("downtime_hours", "sum"),
        total_cost=("maintenance_cost", "sum")
    )
    .reset_index()
)

print(machine_maintenance.to_string(index=False))

MACHINE-WISE MAINTENANCE ANALYSIS
 equipment_id  maintenance_count  total_downtime  total_cost
            1                  2            6.74    12156.97
            2                  2            6.70    10489.34
            3                  3            7.48    21011.44
            4                  2            2.33     3939.47
            5                  4            4.34    14428.55


In [ ]:
#  CORRELATION ANALYSIS

print("CORRELATION ANALYSIS")
print("=" * 70)

# Production + quality + machine information
analysis_df = production.merge(
    quality[["production_id", "defect_count"]],
    on="production_id",
    how="left"
)

# Correlation between important numerical variables
correlation = analysis_df[
    ["units_produced", "units_rejected", "production_time_hours", "defect_count"]
].corr()

print("\nCorrelation Matrix:")
print(correlation.round(2).to_string())

CORRELATION ANALYSIS

Correlation Matrix:
                       units_produced  units_rejected  production_time_hours  defect_count
units_produced                   1.00            0.20                    NaN          0.14
units_rejected                   0.20            1.00                    NaN          0.84
production_time_hours             NaN             NaN                    NaN           NaN
defect_count                     0.14            0.84                    NaN          1.00


In [ ]:
#  EDA CONCLUSIONS

print("EDA CONCLUSIONS")
print("=" * 70)

print("""
1. PRODUCTION:
   - Average production is approximately 911 units per record.
   - Morning shift has the highest average production.
   - Night shift has the lowest average production, but the difference is small.

2. TARGET ACHIEVEMENT:
   - Actual production is compared against machine-wise production targets.
   - Target achievement will be used as an important KPI and ML feature.

3. QUALITY:
   - 73.78% of quality inspections passed.
   - 26.22% of inspections failed.
   - Missing Component has the highest total defect count.
   - Machine 4 has the highest average defects.

4. SENSORS:
   - Average temperature is approximately 70.57°C.
   - Average vibration is approximately 3.22 mm/s.
   - Machine 3 has the highest average temperature and vibration.
   - Sensor readings vary across machines and over time.

5. DOWNTIME:
   - There are 32 downtime events.
   - Average downtime per event is approximately 2.21 hours.
   - Mechanical Failure contributes the highest total downtime.
   - Machine 3 has the highest total downtime.

6. MAINTENANCE:
   - Corrective maintenance has the highest downtime and cost per event.
   - Machine 3 has the highest maintenance downtime and maintenance cost.

7. CORRELATION:
   - Units rejected and defect count have a strong positive correlation (0.84).
   - Units produced and defect count have only a weak correlation.
   - Production time is constant at 8 hours, so it has no meaningful correlation.

8. ML RELEVANCE:
   - Machine 3 shows higher sensor values, downtime and maintenance activity.
   - Sensor readings, quality metrics, downtime and maintenance history
     can therefore be considered useful inputs for future ML modeling.
""")

EDA CONCLUSIONS

1. PRODUCTION:
   - Average production is approximately 911 units per record.
   - Morning shift has the highest average production.
   - Night shift has the lowest average production, but the difference is small.

2. TARGET ACHIEVEMENT:
   - Actual production is compared against machine-wise production targets.
   - Target achievement will be used as an important KPI and ML feature.

3. QUALITY:
   - 73.78% of quality inspections passed.
   - 26.22% of inspections failed.
   - Missing Component has the highest total defect count.
   - Machine 4 has the highest average defects.

4. SENSORS:
   - Average temperature is approximately 70.57°C.
   - Average vibration is approximately 3.22 mm/s.
   - Machine 3 has the highest average temperature and vibration.
   - Sensor readings vary across machines and over time.

5. DOWNTIME:
   - There are 32 downtime events.
   - Average downtime per event is approximately 2.21 hours.
   - Mechanical Failure contributes the highest to

In [ ]:
#  PRODUCTION KPIs

print("PRODUCTION KPIs")
print("=" * 70)

total_production = production["units_produced"].sum()
total_rejected = production["units_rejected"].sum()

total_output = total_production + total_rejected

production_rejection_rate = (
    total_rejected / total_output
) * 100

print("\nTotal Units Produced:")
print(total_production)

print("\nTotal Units Rejected:")
print(total_rejected)

print("\nTotal Production Output:")
print(total_output)

print("\nRejection Rate (%):")
print(round(production_rejection_rate, 2))

PRODUCTION KPIs

Total Units Produced:
410057

Total Units Rejected:
18356

Total Production Output:
428413

Rejection Rate (%):
4.28


In [ ]:
print("TARGET ACHIEVEMENT KPIs")
print("=" * 70)

total_produced = production["units_produced"].sum()
total_target = production_targets["target_quantity"].sum()

overall_target_achievement = (
    total_produced / total_target
) * 100

print("\nTotal Target Quantity:")
print(total_target)

print("\nTotal Units Produced:")
print(total_produced)

print("\nOverall Target Achievement (%):")
print(round(overall_target_achievement, 2))

TARGET ACHIEVEMENT KPIs


NameError: name 'production_targets' is not defined

In [ ]:
print([name for name in globals() if not name.startswith("_")])

['In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'pd', 'mysql', 'sys', 'os', 'get_connection', 'connection', 'cursor', 'tables', 'table', 'data', 'query', 'df', 'summary', 'summary_df', 'important_tables', 'column', 'missing', 'duplicate_count', 'found_invalid', 'col', 'negative_count', 'datetime_cols', 'production', 'quality', 'downtime', 'invalid_downtime', 'targets', 'numeric_target_cols', 'numeric_cols', 'Q1', 'Q3', 'IQR', 'lower_bound', 'upper_bound', 'outliers', 'defect_analysis', 'quality_machine', 'machine_quality', 'sensors', 'sensor_type_summary', 'machine_sensor_summary', 'sensor_trends', 'downtime_reason', 'machine_downtime', 'maintenance', 'maintenance_type_summary', 'machine_maintenance', 'analysis_df', 'correlation', 'total_production', 'total_rejected', 'total_output', 'production_rejection_rate', 'total_produced']


In [ ]:
print("TARGET ACHIEVEMENT KPIs")
print("=" * 70)

total_produced = production["units_produced"].sum()
total_target = targets["target_quantity"].sum()

overall_target_achievement = (
    total_produced / total_target
) * 100

print("\nTotal Target Quantity:")
print(total_target)

print("\nTotal Units Produced:")
print(total_produced)

print("\nOverall Target Achievement (%):")
print(round(overall_target_achievement, 2))

TARGET ACHIEVEMENT KPIs

Total Target Quantity:
419344

Total Units Produced:
410057

Overall Target Achievement (%):
97.79


In [ ]:
print("PRODUCTION COLUMNS:")
print(production.columns.tolist())

print("\nTARGET COLUMNS:")
print(targets.columns.tolist())

PRODUCTION COLUMNS:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours']

TARGET COLUMNS:
['target_id', 'machine_id', 'target_date', 'target_quantity']


In [ ]:
print("ACTUAL PRODUCTION TABLE COLUMNS")
print("=" * 70)

cursor.execute("DESCRIBE production")

for row in cursor.fetchall():
    print(row)

ACTUAL PRODUCTION TABLE COLUMNS
('production_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('machine_id', 'int', 'NO', 'MUL', None, '')
('production_date', 'date', 'NO', '', None, '')
('shift', 'varchar(20)', 'YES', '', None, '')
('units_produced', 'int', 'YES', '', None, '')
('units_rejected', 'int', 'YES', '', None, '')
('production_time_hours', 'decimal(5,2)', 'YES', '', None, '')


In [ ]:
print("\nACTUAL PRODUCTION TARGETS TABLE COLUMNS")
print("=" * 70)

cursor.execute("DESCRIBE production_targets")

for row in cursor.fetchall():
    print(row)


ACTUAL PRODUCTION TARGETS TABLE COLUMNS
('target_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('machine_id', 'int', 'NO', 'MUL', None, '')
('target_date', 'date', 'NO', '', None, '')
('target_quantity', 'int', 'NO', '', None, '')


In [ ]:
#  MACHINE-WISE TARGET ACHIEVEMENT
# ============================================================

print("MACHINE-WISE TARGET ACHIEVEMENT")
print("=" * 70)

machine_production = production.groupby("machine_id")["units_produced"].sum()
machine_target = targets.groupby("machine_id")["target_quantity"].sum()

machine_target_kpi = pd.DataFrame({
    "total_produced": machine_production,
    "total_target": machine_target
})

machine_target_kpi["target_achievement_percent"] = (
    machine_target_kpi["total_produced"] /
    machine_target_kpi["total_target"]
) * 100

machine_target_kpi["performance_status"] = machine_target_kpi[
    "target_achievement_percent"
].apply(
    lambda x: "Above Target" if x > 100
    else "On Target" if x >= 95
    else "Below Target"
)

machine_target_kpi["target_achievement_percent"] = (
    machine_target_kpi["target_achievement_percent"].round(2)
)

print(machine_target_kpi)

MACHINE-WISE TARGET ACHIEVEMENT
            total_produced  total_target  target_achievement_percent  \
machine_id                                                             
1                    84982         90426                       93.98   
2                    81096         80904                      100.24   
3                    71938         72129                       99.74   
4                    88769         90279                       98.33   
5                    83272         85606                       97.27   

           performance_status  
machine_id                     
1                Below Target  
2                Above Target  
3                   On Target  
4                   On Target  
5                   On Target  


In [ ]:
# ============================================================
#  QUALITY KPIs


print("QUALITY KPIs")
print("=" * 70)

total_produced = production["units_produced"].sum()
total_rejected = production["units_rejected"].sum()

quality_defect_rate = (
    total_rejected / total_produced
) * 100

quality_pass_rate = (
    (total_produced - total_rejected) / total_produced
) * 100

print("\nTotal Units Produced:")
print(total_produced)

print("\nTotal Units Rejected:")
print(total_rejected)

print("\nDefect/Rejection Rate (%):")
print(round(quality_defect_rate, 2))

print("\nQuality Pass Rate (%):")
print(round(quality_pass_rate, 2))

QUALITY KPIs

Total Units Produced:
410057

Total Units Rejected:
18356

Defect/Rejection Rate (%):
4.48

Quality Pass Rate (%):
95.52


In [ ]:
#  MACHINE-WISE DEFECT RATE
# ============================================================

print("MACHINE-WISE DEFECT RATE")
print("=" * 70)

machine_quality_kpi = production.groupby("machine_id").agg(
    total_produced=("units_produced", "sum"),
    total_rejected=("units_rejected", "sum")
)

machine_quality_kpi["defect_rate_percent"] = (
    machine_quality_kpi["total_rejected"] /
    machine_quality_kpi["total_produced"]
) * 100

machine_quality_kpi["quality_pass_rate_percent"] = (
    (machine_quality_kpi["total_produced"] -
     machine_quality_kpi["total_rejected"]) /
    machine_quality_kpi["total_produced"]
) * 100

machine_quality_kpi["defect_rate_percent"] = (
    machine_quality_kpi["defect_rate_percent"].round(2)
)

machine_quality_kpi["quality_pass_rate_percent"] = (
    machine_quality_kpi["quality_pass_rate_percent"].round(2)
)

print(machine_quality_kpi)

MACHINE-WISE DEFECT RATE
            total_produced  total_rejected  defect_rate_percent  \
machine_id                                                        
1                    84982            3848                 4.53   
2                    81096            3751                 4.63   
3                    71938            3213                 4.47   
4                    88769            4185                 4.71   
5                    83272            3359                 4.03   

            quality_pass_rate_percent  
machine_id                             
1                               95.47  
2                               95.37  
3                               95.53  
4                               95.29  
5                               95.97  


In [ ]:

#  CHECK DOWNTIME DATA
# ============================================================

print("DOWNTIME COLUMNS")
print("=" * 70)

print(downtime.columns.tolist())

DOWNTIME COLUMNS
['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']


In [ ]:
#  DOWNTIME KPIs
# ============================================================

print("DOWNTIME KPIs")
print("=" * 70)

total_downtime_hours = downtime["downtime_hours"].sum()

average_downtime_hours = downtime["downtime_hours"].mean()

total_downtime_events = downtime["downtime_id"].count()

print("\nTotal Downtime (Hours):")
print(round(total_downtime_hours, 2))

print("\nAverage Downtime per Event (Hours):")
print(round(average_downtime_hours, 2))

print("\nTotal Downtime Events:")
print(total_downtime_events)

DOWNTIME KPIs

Total Downtime (Hours):
70.86

Average Downtime per Event (Hours):
2.21

Total Downtime Events:
32


In [ ]:

# MACHINE-WISE DOWNTIME
# ============================================================

print("MACHINE-WISE DOWNTIME")
print("=" * 70)

machine_downtime_kpi = downtime.groupby("machine_id").agg(
    total_downtime_hours=("downtime_hours", "sum"),
    downtime_events=("downtime_id", "count")
)

machine_downtime_kpi["average_downtime_hours"] = (
    machine_downtime_kpi["total_downtime_hours"] /
    machine_downtime_kpi["downtime_events"]
)

machine_downtime_kpi = machine_downtime_kpi.round(2)

print(machine_downtime_kpi)

MACHINE-WISE DOWNTIME
            total_downtime_hours  downtime_events  average_downtime_hours
machine_id                                                               
1                           7.06                4                    1.76
2                          15.49                6                    2.58
3                          26.39               13                    2.03
4                          16.30                6                    2.72
5                           5.62                3                    1.87


In [ ]:

# KPI 5A: CHECK PRODUCTION TIME DATA
# ============================================================

print("PRODUCTION TIME SUMMARY")
print("=" * 70)

print("\nProduction Time Statistics:")
print(production["production_time_hours"].describe())

print("\nTotal Production Time (Hours):")
print(round(production["production_time_hours"].sum(), 2))

PRODUCTION TIME SUMMARY

Production Time Statistics:
count    450.0
mean       8.0
std        0.0
min        8.0
25%        8.0
50%        8.0
75%        8.0
max        8.0
Name: production_time_hours, dtype: float64

Total Production Time (Hours):
3600.0


In [ ]:
# ============================================================
# MACHINE UTILIZATION
# ============================================================

print("MACHINE UTILIZATION KPIs")
print("=" * 70)

# Total available production hours for each machine
machine_available_hours = production.groupby("machine_id")[
    "production_time_hours"
].sum()

# Total downtime hours for each machine
machine_downtime_hours = downtime.groupby("machine_id")[
    "downtime_hours"
].sum()

# Combine available hours and downtime
machine_utilization_kpi = pd.DataFrame({
    "available_hours": machine_available_hours,
    "downtime_hours": machine_downtime_hours
}).fillna(0)

# Calculate actual operating hours
machine_utilization_kpi["operating_hours"] = (
    machine_utilization_kpi["available_hours"] -
    machine_utilization_kpi["downtime_hours"]
)

# Calculate utilization percentage
machine_utilization_kpi["utilization_percent"] = (
    machine_utilization_kpi["operating_hours"] /
    machine_utilization_kpi["available_hours"]
) * 100

machine_utilization_kpi["utilization_percent"] = (
    machine_utilization_kpi["utilization_percent"].round(2)
)

print(machine_utilization_kpi)

MACHINE UTILIZATION KPIs
            available_hours  downtime_hours  operating_hours  \
machine_id                                                     
1                     720.0            7.06           712.94   
2                     720.0           15.49           704.51   
3                     720.0           26.39           693.61   
4                     720.0           16.30           703.70   
5                     720.0            5.62           714.38   

            utilization_percent  
machine_id                       
1                         99.02  
2                         97.85  
3                         96.33  
4                         97.74  
5                         99.22  


In [ ]:

#  CHECK AVAILABLE DATA
# ============================================================

print("AVAILABLE DATAFRAMES")
print("=" * 70)

dataframes = {
    "production": production,
    "targets": targets,
    "quality": quality,
    "downtime": downtime,
    "sensors": sensors,
    "maintenance": maintenance
}

for name, dataframe in dataframes.items():
    print(f"\n{name}:")
    print(f"Rows    : {dataframe.shape[0]}")
    print(f"Columns : {dataframe.shape[1]}")
    print(f"Columns : {dataframe.columns.tolist()}")

AVAILABLE DATAFRAMES


NameError: name 'sensors' is not defined

In [ ]:
# ============================================================
# CHECK LOADED DATAFRAMES
# ============================================================

print("LOADED DATAFRAMES")
print("=" * 70)

for name in ["production", "targets", "quality", "downtime", "sensors", "maintenance"]:
    print(f"{name}: {name in globals()}")

LOADED DATAFRAMES
production: True
targets: True
quality: True
downtime: True
sensors: False
maintenance: False


In [ ]:
# ============================================================
# RELOAD MISSING DATAFRAMES
# ============================================================

print("RELOADING SENSOR AND MAINTENANCE DATA")
print("=" * 70)

# Load sensors table
query = "SELECT * FROM sensors"
sensors = pd.read_sql(query, connection)

# Load maintenance table
query = "SELECT * FROM maintenance"
maintenance = pd.read_sql(query, connection)

print("\nSensors:")
print("Rows:", sensors.shape[0])
print("Columns:", sensors.columns.tolist())

print("\nMaintenance:")
print("Rows:", maintenance.shape[0])
print("Columns:", maintenance.columns.tolist())

RELOADING SENSOR AND MAINTENANCE DATA

Sensors:
Rows: 900
Columns: ['sensor_id', 'machine_id', 'sensor_type', 'sensor_value', 'unit', 'recorded_at']

Maintenance:
Rows: 13
Columns: ['maintenance_id', 'equipment_id', 'maintenance_date', 'maintenance_type', 'maintenance_status', 'downtime_hours', 'maintenance_cost']


C:\Users\Sai Sanjana S\AppData\Local\Temp\ipykernel_29440\4204890061.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sensors = pd.read_sql(query, connection)
C:\Users\Sai Sanjana S\AppData\Local\Temp\ipykernel_29440\4204890061.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  maintenance = pd.read_sql(query, connection)


In [ ]:

 #table relationships


print("TABLE RELATIONSHIPS")
print("=" * 70)

print("\nPRODUCTION:")
print(production.columns.tolist())

print("\nTARGETS:")
print(targets.columns.tolist())

print("\nQUALITY:")
print(quality.columns.tolist())

print("\nDOWNTIME:")
print(downtime.columns.tolist())

print("\nSENSORS:")
print(sensors.columns.tolist())

print("\nMAINTENANCE:")
print(maintenance.columns.tolist())

TABLE RELATIONSHIPS

PRODUCTION:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours']

TARGETS:
['target_id', 'machine_id', 'target_date', 'target_quantity']

QUALITY:
['quality_id', 'production_id', 'inspection_date', 'defect_type', 'defect_count', 'quality_status']

DOWNTIME:
['downtime_id', 'machine_id', 'downtime_start', 'downtime_end', 'downtime_reason', 'downtime_hours']

SENSORS:
['sensor_id', 'machine_id', 'sensor_type', 'sensor_value', 'unit', 'recorded_at']

MAINTENANCE:
['maintenance_id', 'equipment_id', 'maintenance_date', 'maintenance_type', 'maintenance_status', 'downtime_hours', 'maintenance_cost']


In [ ]:
# ============================================================
#  PRODUCTION + TARGETS


print("BUILDING PRODUCTION-TARGET DATASET")
print("=" * 70)

# Merge production with production targets
master_df = production.merge(
    targets,
    left_on=["machine_id", "production_date"],
    right_on=["machine_id", "target_date"],
    how="left"
)

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nMaster Dataset Columns:")
print(master_df.columns.tolist())

print("\nMissing Target Values:")
print(master_df["target_quantity"].isna().sum())

BUILDING PRODUCTION-TARGET DATASET

Master Dataset Shape:
(1350, 10)

Master Dataset Columns:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity']

Missing Target Values:
0


In [ ]:
# CHECKING TARGET DUPLICATES
# ============================================================

print("CHECKING TARGET DUPLICATES")
print("=" * 70)

target_duplicates = targets[
    targets.duplicated(
        subset=["machine_id", "target_date"],
        keep=False
    )
].sort_values(["machine_id", "target_date"])

print("\nDuplicate machine-date target records:")
print(target_duplicates)

print("\nNumber of duplicate rows:")
print(len(target_duplicates))

CHECKING TARGET DUPLICATES

Duplicate machine-date target records:
     target_id  machine_id target_date  target_quantity
0            1           1  2026-07-20              998
1            2           1  2026-07-20             1012
2            3           1  2026-07-20             1003
15          16           1  2026-07-21              974
16          17           1  2026-07-21              978
..         ...         ...         ...              ...
433        434           5  2026-08-17              936
434        435           5  2026-08-17              935
447        448           5  2026-08-18              947
448        449           5  2026-08-18              950
449        450           5  2026-08-18              947

[450 rows x 4 columns]

Number of duplicate rows:
450


In [ ]:
# ============================================================
# INSPECTING TARGET-SHIFT STRUCTURE
# ============================================================

print("TARGET AND PRODUCTION STRUCTURE")
print("=" * 70)

print("\nProduction records per machine/date:")
print(
    production.groupby(
        ["machine_id", "production_date"]
    ).size().value_counts()
)

print("\nTarget records per machine/date:")
print(
    targets.groupby(
        ["machine_id", "target_date"]
    ).size().value_counts()
)

print("\nSample production records:")
print(
    production[
        ["machine_id", "production_date", "shift", "units_produced"]
    ].head(15)
)

print("\nSample target records:")
print(
    targets[
        ["machine_id", "target_date", "target_quantity"]
    ].head(15)
)

TARGET AND PRODUCTION STRUCTURE

Production records per machine/date:
3    150
Name: count, dtype: int64

Target records per machine/date:
3    150
Name: count, dtype: int64

Sample production records:
    machine_id production_date      shift  units_produced
0            1      2026-07-20    Morning             969
1            1      2026-07-20  Afternoon             944
2            1      2026-07-20      Night             940
3            2      2026-07-20    Morning             890
4            2      2026-07-20  Afternoon             963
5            2      2026-07-20      Night             930
6            3      2026-07-20    Morning             781
7            3      2026-07-20  Afternoon             821
8            3      2026-07-20      Night             809
9            4      2026-07-20    Morning             903
10           4      2026-07-20  Afternoon             939
11           4      2026-07-20      Night             992
12           5      2026-07-20    Morning   

In [ ]:
#CORRECT TARGET MAPPING
# ============================================================

print("CREATING CORRECT PRODUCTION-TARGET DATASET")
print("=" * 70)

# Create an order number within each machine/date group
production_temp = production.copy()
targets_temp = targets.copy()

production_temp["shift_order"] = (
    production_temp
    .groupby(["machine_id", "production_date"])
    .cumcount()
)

targets_temp["shift_order"] = (
    targets_temp
    .groupby(["machine_id", "target_date"])
    .cumcount()
)

# Merge using machine + date + order
master_df = production_temp.merge(
    targets_temp,
    left_on=["machine_id", "production_date", "shift_order"],
    right_on=["machine_id", "target_date", "shift_order"],
    how="left"
)

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nMissing Target Values:")
print(master_df["target_quantity"].isna().sum())

print("\nShift Distribution:")
print(master_df["shift"].value_counts())

print("\nMaster Dataset Preview:")
print(
    master_df[
        ["production_id", "machine_id", "production_date",
         "shift", "units_produced", "target_quantity"]
    ].head(15).to_string(index=False)
)

CREATING CORRECT PRODUCTION-TARGET DATASET

Master Dataset Shape:
(450, 11)

Missing Target Values:
0

Shift Distribution:
shift
Morning      150
Afternoon    150
Night        150
Name: count, dtype: int64

Master Dataset Preview:
 production_id  machine_id production_date     shift  units_produced  target_quantity
             1           1      2026-07-20   Morning             969              998
             2           1      2026-07-20 Afternoon             944             1012
             3           1      2026-07-20     Night             940             1003
             4           2      2026-07-20   Morning             890              873
             5           2      2026-07-20 Afternoon             963              907
             6           2      2026-07-20     Night             930              912
             7           3      2026-07-20   Morning             781              811
             8           3      2026-07-20 Afternoon             821             

In [ ]:
# ============================================================
#  PRODUCTION FEATURES

print("CREATING PRODUCTION FEATURES")
print("=" * 70)

# Target achievement
master_df["target_achievement_percent"] = (
    master_df["units_produced"] /
    master_df["target_quantity"]
) * 100

# Total production output
master_df["total_output"] = (
    master_df["units_produced"] +
    master_df["units_rejected"]
)

# Rejection rate
master_df["rejection_rate_percent"] = (
    master_df["units_rejected"] /
    master_df["total_output"]
) * 100

# Production efficiency
master_df["production_rate_per_hour"] = (
    master_df["units_produced"] /
    master_df["production_time_hours"]
)

# Round calculated features
master_df["target_achievement_percent"] = (
    master_df["target_achievement_percent"].round(2)
)

master_df["rejection_rate_percent"] = (
    master_df["rejection_rate_percent"].round(2)
)

master_df["production_rate_per_hour"] = (
    master_df["production_rate_per_hour"].round(2)
)

print("\nProduction Features Created:")
print([
    "target_achievement_percent",
    "total_output",
    "rejection_rate_percent",
    "production_rate_per_hour"
])

print("\nFeature Preview:")
print(
    master_df[
        [
            "machine_id",
            "shift",
            "units_produced",
            "target_quantity",
            "target_achievement_percent",
            "units_rejected",
            "rejection_rate_percent",
            "production_rate_per_hour"
        ]
    ].head(10)
)

CREATING PRODUCTION FEATURES

Production Features Created:
['target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour']

Feature Preview:
   machine_id      shift  units_produced  target_quantity  \
0           1    Morning             969              998   
1           1  Afternoon             944             1012   
2           1      Night             940             1003   
3           2    Morning             890              873   
4           2  Afternoon             963              907   
5           2      Night             930              912   
6           3    Morning             781              811   
7           3  Afternoon             821              821   
8           3      Night             809              816   
9           4    Morning             903             1009   

   target_achievement_percent  units_rejected  rejection_rate_percent  \
0                       97.09              59                    5.74   
1   

In [ ]:
# QUALITY FEATURES
# ============================================================

print("CREATING QUALITY FEATURES")
print("=" * 70)

# Aggregate quality data by production record
quality_features = quality.groupby("production_id").agg(
    total_defects=("defect_count", "sum"),
    defect_type_count=("defect_type", "nunique")
).reset_index()

# Merge quality features into master dataset
master_df = master_df.merge(
    quality_features,
    on="production_id",
    how="left"
)

# Production records without quality defects
master_df["total_defects"] = (
    master_df["total_defects"].fillna(0)
)

master_df["defect_type_count"] = (
    master_df["defect_type_count"].fillna(0)
)

# Quality defect rate
master_df["quality_defect_rate_percent"] = (
    master_df["total_defects"] /
    master_df["total_output"]
) * 100

master_df["quality_defect_rate_percent"] = (
    master_df["quality_defect_rate_percent"].round(2)
)

print("\nQuality Features Created:")
print([
    "total_defects",
    "defect_type_count",
    "quality_defect_rate_percent"
])

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nMissing Quality Values:")
print(
    master_df[
        ["total_defects", "defect_type_count"]
    ].isna().sum()
)

print("\nFeature Preview:")
print(
    master_df[
        [
            "production_id",
            "total_output",
            "total_defects",
            "defect_type_count",
            "quality_defect_rate_percent"
        ]
    ].head(10)
)

CREATING QUALITY FEATURES

Quality Features Created:
['total_defects', 'defect_type_count', 'quality_defect_rate_percent']

Master Dataset Shape:
(450, 18)

Missing Quality Values:
total_defects        0
defect_type_count    0
dtype: int64

Feature Preview:
   production_id  total_output  total_defects  defect_type_count  \
0              1          1028             17                  1   
1              2           992              9                  1   
2              3           953              3                  1   
3              4           952             24                  1   
4              5           974              3                  1   
5              6          1002             18                  1   
6              7           798              5                  1   
7              8           839              2                  1   
8              9           841             11                  1   
9             10           930              8                 

In [ ]:

#  DOWNTIME FEATURES
# ============================================================

print("CREATING DOWNTIME FEATURES")
print("=" * 70)

# Aggregate downtime by machine
downtime_features = downtime.groupby("machine_id").agg(
    total_downtime_hours=("downtime_hours", "sum"),
    downtime_event_count=("downtime_id", "count"),
    average_downtime_hours=("downtime_hours", "mean")
).reset_index()

# Merge downtime features into master dataset
master_df = master_df.merge(
    downtime_features,
    on="machine_id",
    how="left"
)

# Fill missing downtime values
master_df["total_downtime_hours"] = (
    master_df["total_downtime_hours"].fillna(0)
)

master_df["downtime_event_count"] = (
    master_df["downtime_event_count"].fillna(0)
)

master_df["average_downtime_hours"] = (
    master_df["average_downtime_hours"].fillna(0)
)

# Total available production hours per machine
available_hours = (
    production.groupby("machine_id")["production_time_hours"]
    .sum()
    .reset_index()
    .rename(columns={
        "production_time_hours": "available_hours"
    })
)

# Merge available hours
master_df = master_df.merge(
    available_hours,
    on="machine_id",
    how="left"
)

# Operating hours
master_df["operating_hours"] = (
    master_df["available_hours"] -
    master_df["total_downtime_hours"]
)

# Utilization
master_df["utilization_percent"] = (
    master_df["operating_hours"] /
    master_df["available_hours"]
) * 100

# Round values
master_df["average_downtime_hours"] = (
    master_df["average_downtime_hours"].round(2)
)

master_df["operating_hours"] = (
    master_df["operating_hours"].round(2)
)

master_df["utilization_percent"] = (
    master_df["utilization_percent"].round(2)
)

print("\nDowntime Features Created:")
print([
    "total_downtime_hours",
    "downtime_event_count",
    "average_downtime_hours",
    "available_hours",
    "operating_hours",
    "utilization_percent"
])

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nFeature Preview:")
print(
    master_df[
        [
            "machine_id",
            "total_downtime_hours",
            "downtime_event_count",
            "average_downtime_hours",
            "available_hours",
            "operating_hours",
            "utilization_percent"
        ]
    ].drop_duplicates("machine_id")
)

CREATING DOWNTIME FEATURES

Downtime Features Created:
['total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent']

Master Dataset Shape:
(450, 24)

Feature Preview:
    machine_id  total_downtime_hours  downtime_event_count  \
0            1                  7.06                     4   
3            2                 15.49                     6   
6            3                 26.39                    13   
9            4                 16.30                     6   
12           5                  5.62                     3   

    average_downtime_hours  available_hours  operating_hours  \
0                     1.76            720.0           712.94   
3                     2.58            720.0           704.51   
6                     2.03            720.0           693.61   
9                     2.72            720.0           703.70   
12                    1.87            720.0           714.38   



In [ ]:
# SENSOR FEATURES
# ============================================================

print("CREATING SENSOR FEATURES")
print("=" * 70)

# Aggregate sensor readings by machine
sensor_features = sensors.groupby("machine_id").agg(
    sensor_value_mean=("sensor_value", "mean"),
    sensor_value_min=("sensor_value", "min"),
    sensor_value_max=("sensor_value", "max"),
    sensor_value_std=("sensor_value", "std"),
    sensor_reading_count=("sensor_id", "count")
).reset_index()

# Merge sensor features into master dataset
master_df = master_df.merge(
    sensor_features,
    on="machine_id",
    how="left"
)

# Fill missing values
sensor_columns = [
    "sensor_value_mean",
    "sensor_value_min",
    "sensor_value_max",
    "sensor_value_std",
    "sensor_reading_count"
]

master_df[sensor_columns] = (
    master_df[sensor_columns].fillna(0)
)

# Round sensor values
master_df["sensor_value_mean"] = (
    master_df["sensor_value_mean"].round(2)
)

master_df["sensor_value_min"] = (
    master_df["sensor_value_min"].round(2)
)

master_df["sensor_value_max"] = (
    master_df["sensor_value_max"].round(2)
)

master_df["sensor_value_std"] = (
    master_df["sensor_value_std"].round(2)
)

print("\nSensor Features Created:")
print(sensor_columns)

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nFeature Preview:")
print(
    master_df[
        [
            "machine_id",
            "sensor_value_mean",
            "sensor_value_min",
            "sensor_value_max",
            "sensor_value_std",
            "sensor_reading_count"
        ]
    ].drop_duplicates("machine_id")
)

CREATING SENSOR FEATURES

Sensor Features Created:
['sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count']

Master Dataset Shape:
(450, 29)

Feature Preview:
    machine_id  sensor_value_mean  sensor_value_min  sensor_value_max  \
0            1              36.48              1.31             76.56   
3            2              35.47              1.60             75.40   
6            3              43.64              3.76             88.87   
9            4              33.98              1.24             75.49   
12           5              34.90              1.60             75.02   

    sensor_value_std  sensor_reading_count  
0              33.65                   180  
3              32.86                   180  
6              38.73                   180  
9              31.59                   180  
12             32.34                   180  


In [ ]:
#MAINTENANCE FEATURES
# ============================================================

print("CREATING MAINTENANCE FEATURES")
print("=" * 70)

# Aggregate maintenance data by equipment
maintenance_features = maintenance.groupby("equipment_id").agg(
    maintenance_event_count=("maintenance_id", "count"),
    total_maintenance_downtime=("downtime_hours", "sum"),
    average_maintenance_downtime=("downtime_hours", "mean"),
    total_maintenance_cost=("maintenance_cost", "sum")
).reset_index()

# Rename equipment_id to machine_id for merging
maintenance_features = maintenance_features.rename(
    columns={"equipment_id": "machine_id"}
)

# Count completed maintenance events
completed_maintenance = (
    maintenance[maintenance["maintenance_status"].str.lower() == "completed"]
    .groupby("equipment_id")
    .size()
    .reset_index(name="completed_maintenance_count")
)

completed_maintenance = completed_maintenance.rename(
    columns={"equipment_id": "machine_id"}
)

# Merge completed maintenance count
maintenance_features = maintenance_features.merge(
    completed_maintenance,
    on="machine_id",
    how="left"
)

# Merge maintenance features into master dataset
master_df = master_df.merge(
    maintenance_features,
    on="machine_id",
    how="left"
)

# Fill missing maintenance values
maintenance_columns = [
    "maintenance_event_count",
    "total_maintenance_downtime",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count"
]

master_df[maintenance_columns] = (
    master_df[maintenance_columns].fillna(0)
)

# Round values
master_df["total_maintenance_downtime"] = (
    master_df["total_maintenance_downtime"].round(2)
)

master_df["average_maintenance_downtime"] = (
    master_df["average_maintenance_downtime"].round(2)
)

master_df["total_maintenance_cost"] = (
    master_df["total_maintenance_cost"].round(2)
)

print("\nMaintenance Features Created:")
print(maintenance_columns)

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nFeature Preview:")
print(
    master_df[
        [
            "machine_id",
            "maintenance_event_count",
            "total_maintenance_downtime",
            "average_maintenance_downtime",
            "total_maintenance_cost",
            "completed_maintenance_count"
        ]
    ].drop_duplicates("machine_id")
)

CREATING MAINTENANCE FEATURES

Maintenance Features Created:
['maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count']

Master Dataset Shape:
(450, 34)

Feature Preview:
    machine_id  maintenance_event_count  total_maintenance_downtime  \
0            1                        2                        6.74   
3            2                        2                        6.70   
6            3                        3                        7.48   
9            4                        2                        2.33   
12           5                        4                        4.34   

    average_maintenance_downtime  total_maintenance_cost  \
0                           3.37                12156.97   
3                           3.35                10489.34   
6                           2.49                21011.44   
9                           1.16                 3939.47   
12            

In [ ]:
# ============================================================
#  TIME AND SHIFT FEATURES
# ============================================================

print("CREATING TIME AND SHIFT FEATURES")
print("=" * 70)

# Make sure production date is datetime
master_df["production_date"] = pd.to_datetime(
    master_df["production_date"]
)

# Date-based features
master_df["production_day"] = (
    master_df["production_date"].dt.day
)

master_df["production_month"] = (
    master_df["production_date"].dt.month
)

master_df["production_day_of_week"] = (
    master_df["production_date"].dt.dayofweek
)

# Weekend indicator
master_df["is_weekend"] = (
    master_df["production_day_of_week"] >= 5
).astype(int)

# Shift encoding
shift_mapping = {
    "Morning": 0,
    "Afternoon": 1,
    "Night": 2
}

master_df["shift_encoded"] = (
    master_df["shift"].map(shift_mapping)
)

print("\nTime and Shift Features Created:")
print([
    "production_day",
    "production_month",
    "production_day_of_week",
    "is_weekend",
    "shift_encoded"
])

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nFeature Preview:")
print(
    master_df[
        [
            "production_date",
            "shift",
            "production_day",
            "production_month",
            "production_day_of_week",
            "is_weekend",
            "shift_encoded"
        ]
    ].head(10)
)

CREATING TIME AND SHIFT FEATURES

Time and Shift Features Created:
['production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded']

Master Dataset Shape:
(450, 39)

Feature Preview:
  production_date      shift  production_day  production_month  \
0      2026-07-20    Morning              20                 7   
1      2026-07-20  Afternoon              20                 7   
2      2026-07-20      Night              20                 7   
3      2026-07-20    Morning              20                 7   
4      2026-07-20  Afternoon              20                 7   
5      2026-07-20      Night              20                 7   
6      2026-07-20    Morning              20                 7   
7      2026-07-20  Afternoon              20                 7   
8      2026-07-20      Night              20                 7   
9      2026-07-20    Morning              20                 7   

   production_day_of_week  is_weekend  shift_encoded  
0   

In [ ]:

#  HISTORICAL FEATURES


print("CREATING HISTORICAL PRODUCTION FEATURES")
print("=" * 70)

# Define shift order
shift_order = {
    "Morning": 0,
    "Afternoon": 1,
    "Night": 2
}

# Create temporary shift order for sorting
master_df["shift_order"] = (
    master_df["shift"].map(shift_order)
)

# Sort chronologically within each machine
master_df = master_df.sort_values(
    ["machine_id", "production_date", "shift_order"]
).reset_index(drop=True)

# Previous production
master_df["previous_units_produced"] = (
    master_df
    .groupby("machine_id")["units_produced"]
    .shift(1)
)

# Previous rejection rate
master_df["previous_rejection_rate"] = (
    master_df
    .groupby("machine_id")["rejection_rate_percent"]
    .shift(1)
)

# Rolling average production over previous 3 records
master_df["rolling_3_production_avg"] = (
    master_df
    .groupby("machine_id")["units_produced"]
    .transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
)

# Rolling average rejection rate over previous 3 records
master_df["rolling_3_rejection_avg"] = (
    master_df
    .groupby("machine_id")["rejection_rate_percent"]
    .transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
)

# Round values
master_df["rolling_3_production_avg"] = (
    master_df["rolling_3_production_avg"].round(2)
)

master_df["previous_rejection_rate"] = (
    master_df["previous_rejection_rate"].round(2)
)

master_df["rolling_3_rejection_avg"] = (
    master_df["rolling_3_rejection_avg"].round(2)
)

print("\nHistorical Features Created:")
print([
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
])

print("\nMaster Dataset Shape:")
print(master_df.shape)

print("\nHistorical Feature Preview:")
print(
    master_df[
        [
            "machine_id",
            "production_date",
            "shift",
            "units_produced",
            "previous_units_produced",
            "previous_rejection_rate",
            "rolling_3_production_avg",
            "rolling_3_rejection_avg"
        ]
    ].head(15).to_string(index=False)
)

CREATING HISTORICAL PRODUCTION FEATURES

Historical Features Created:
['previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']

Master Dataset Shape:
(450, 43)

Historical Feature Preview:
 machine_id production_date     shift  units_produced  previous_units_produced  previous_rejection_rate  rolling_3_production_avg  rolling_3_rejection_avg
          1      2026-07-20   Morning             969                      NaN                      NaN                       NaN                      NaN
          1      2026-07-20 Afternoon             944                    969.0                     5.74                    969.00                     5.74
          1      2026-07-20     Night             940                    944.0                     4.84                    956.50                     5.29
          1      2026-07-21   Morning             954                    940.0                     1.36                    951.00         

In [ ]:
#FINAL VALIDATION
# ============================================================

print("FINAL FEATURE DATASET VALIDATION")
print("=" * 70)

# Remove temporary sorting column
master_df = master_df.drop(columns=["shift_order"])

# Check shape
print("\nDataset Shape:")
print(master_df.shape)

# Check duplicate production records
duplicate_production_ids = (
    master_df["production_id"].duplicated().sum()
)

print("\nDuplicate Production IDs:")
print(duplicate_production_ids)

# Check missing values
missing_values = master_df.isnull().sum()

print("\nColumns With Missing Values:")
print(missing_values[missing_values > 0])

# Check infinite values
numeric_data = master_df.select_dtypes(include="number")

infinite_values = (
    numeric_data.isin([float("inf"), float("-inf")])
    .sum()
    .sum()
)

print("\nInfinite Values:")
print(infinite_values)

# Number of numeric features
print("\nNumeric Columns:")
print(len(numeric_data.columns))

# Final columns
print("\nFinal Feature Columns:")
print(master_df.columns.tolist())

FINAL FEATURE DATASET VALIDATION

Dataset Shape:
(450, 42)

Duplicate Production IDs:
0

Columns With Missing Values:
previous_units_produced     5
previous_rejection_rate     5
rolling_3_production_avg    5
rolling_3_rejection_avg     5
dtype: int64

Infinite Values:
0

Numeric Columns:
39

Final Feature Columns:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance

In [ ]:

#  SAVE FEATURE DATASET

print("SAVING FEATURE-ENGINEERED DATASET")
print("=" * 70)

# Create output directory
import os

os.makedirs("../data/processed", exist_ok=True)

# Save feature-engineered dataset
feature_file = "../data/processed/feature_engineered_data.csv"

master_df.to_csv(
    feature_file,
    index=False
)

print("\nFeature dataset saved successfully!")
print("File:", feature_file)
print("Rows:", master_df.shape[0])
print("Columns:", master_df.shape[1])

SAVING FEATURE-ENGINEERED DATASET

Feature dataset saved successfully!
File: ../data/processed/feature_engineered_data.csv
Rows: 450
Columns: 42


In [ ]:
# LOAD DATA
# ============================================================

import pandas as pd

feature_data = pd.read_csv(
    "../data/processed/feature_engineered_data.csv"
)

print("ML DATASET LOADED")
print("=" * 70)

print("Shape:", feature_data.shape)

print("\nColumns:")
print(feature_data.columns.tolist())

print("\nMissing Values:")
print(feature_data.isnull().sum()[feature_data.isnull().sum() > 0])

ML DATASET LOADED
Shape: (450, 42)

Columns:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']

Missing Values:
previous_un

In [ ]:
# COLUMN CATEGORIZATION
# ============================================================

print("COLUMN CATEGORIZATION")
print("=" * 70)

print("\nIDENTIFIER COLUMNS:")
print([
    "production_id",
    "target_id"
])

print("\nDATE / TIME COLUMNS:")
print([
    "production_date",
    "target_date"
])

print("\nPRODUCTION COLUMNS:")
print([
    "units_produced",
    "units_rejected",
    "production_time_hours"
])

print("\nTARGET / PERFORMANCE COLUMNS:")
print([
    "target_quantity",
    "target_achievement_percent",
    "total_output",
    "rejection_rate_percent",
    "production_rate_per_hour"
])

print("\nQUALITY COLUMNS:")
print([
    "total_defects",
    "defect_type_count",
    "quality_defect_rate_percent"
])

print("\nDOWNTIME / UTILIZATION COLUMNS:")
print([
    "total_downtime_hours",
    "downtime_event_count",
    "average_downtime_hours",
    "available_hours",
    "operating_hours",
    "utilization_percent"
])

print("\nSENSOR COLUMNS:")
print([
    "sensor_value_mean",
    "sensor_value_min",
    "sensor_value_max",
    "sensor_value_std",
    "sensor_reading_count"
])

print("\nMAINTENANCE COLUMNS:")
print([
    "maintenance_event_count",
    "total_maintenance_downtime",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count"
])

print("\nTIME / SHIFT FEATURES:")
print([
    "production_day",
    "production_month",
    "production_day_of_week",
    "is_weekend",
    "shift_encoded"
])

print("\nHISTORICAL FEATURES:")
print([
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
])

COLUMN CATEGORIZATION

IDENTIFIER COLUMNS:
['production_id', 'target_id']

DATE / TIME COLUMNS:
['production_date', 'target_date']

PRODUCTION COLUMNS:
['units_produced', 'units_rejected', 'production_time_hours']

TARGET / PERFORMANCE COLUMNS:
['target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour']

QUALITY COLUMNS:
['total_defects', 'defect_type_count', 'quality_defect_rate_percent']

DOWNTIME / UTILIZATION COLUMNS:
['total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent']

SENSOR COLUMNS:
['sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count']

MAINTENANCE COLUMNS:
['maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count']

TIME / SHIFT FEATURES:
['production_day', 'production_month', 'production_day_of_week

In [ ]:
# DATA TYPE & MISSING CHECK
# ============================================================

print("ML DATASET PRE-CHECK")
print("=" * 70)

print("\nDATA TYPES:")
print(feature_data.dtypes)

print("\nMISSING VALUES:")
missing = feature_data.isnull().sum()
print(missing[missing > 0])

print("\nDUPLICATE PRODUCTION IDs:")
print(feature_data["production_id"].duplicated().sum())

print("\nTARGET STATISTICS:")
print(feature_data["units_produced"].describe())

ML DATASET PRE-CHECK

DATA TYPES:
production_id                     int64
machine_id                        int64
production_date                     str
shift                               str
units_produced                    int64
units_rejected                    int64
production_time_hours           float64
target_id                         int64
target_date                         str
target_quantity                   int64
target_achievement_percent      float64
total_output                      int64
rejection_rate_percent          float64
production_rate_per_hour        float64
total_defects                     int64
defect_type_count                 int64
quality_defect_rate_percent     float64
total_downtime_hours            float64
downtime_event_count              int64
average_downtime_hours          float64
available_hours                 float64
operating_hours                 float64
utilization_percent             float64
sensor_value_mean               float64
sensor

In [ ]:
# ============================================================

# CREATING PRODUCTION PREDICTION DATASET
# ============================================================

# Target variable
target_column = "units_produced"

# Features that should NOT be used for production prediction
columns_to_remove = [
    "production_id",
    "target_id",
    "production_date",
    "target_date",
    
    # Target itself
    "units_produced",
    
    # Features calculated using the target
    "target_achievement_percent",
    "total_output",
    "rejection_rate_percent",
    "production_rate_per_hour"
]

# Create X
X_production = feature_data.drop(
    columns=columns_to_remove
)

# Create y
y_production = feature_data[target_column]

print("PRODUCTION ML DATASET")
print("=" * 70)

print("\nX Shape:")
print(X_production.shape)

print("\ny Shape:")
print(y_production.shape)

print("\nX Columns:")
print(X_production.columns.tolist())

print("\nTarget:")
print(target_column)

print("\nTarget Statistics:")
print(y_production.describe())

PRODUCTION ML DATASET

X Shape:
(450, 33)

y Shape:
(450,)

X Columns:
['machine_id', 'shift', 'units_rejected', 'production_time_hours', 'target_quantity', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']

Target:
units_produced

Target Statistics:
count     450.000000
mean      911.237778
std        73.708261
min       670.000000
25%       868.000000
50%       921.000000
75%       961.

In [ ]:
# 
# CREATING PRODUCTION PREDICTION DATASET
# ============================================================

# Target variable
target_column = "units_produced"

# Features that should NOT be used for production prediction
columns_to_remove = [
    "production_id",
    "target_id",
    "production_date",
    "target_date",
    
    # Target itself
    "units_produced",
    
    # Features calculated using the target
    "target_achievement_percent",
    "total_output",
    "rejection_rate_percent",
    "production_rate_per_hour"
]

# Create X
X_production = feature_data.drop(
    columns=columns_to_remove
)

# Create y
y_production = feature_data[target_column]

print("PRODUCTION ML DATASET")
print("=" * 70)

print("\nX Shape:")
print(X_production.shape)

print("\ny Shape:")
print(y_production.shape)

print("\nX Columns:")
print(X_production.columns.tolist())

print("\nTarget:")
print(target_column)

print("\nTarget Statistics:")
print(y_production.describe())

PRODUCTION ML DATASET

X Shape:
(450, 33)

y Shape:
(450,)

X Columns:
['machine_id', 'shift', 'units_rejected', 'production_time_hours', 'target_quantity', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']

Target:
units_produced

Target Statistics:
count     450.000000
mean      911.237778
std        73.708261
min       670.000000
25%       868.000000
50%       921.000000
75%       961.

In [ ]:

# CLEAN X FOR MACHINE LEARNING
# ============================================================

# Make a copy so the original X remains unchanged
X_production_clean = X_production.copy()

# Remove string-based shift column
# shift_encoded is already available
X_production_clean = X_production_clean.drop(columns=["shift"])

# Check missing values before handling
print("MISSING VALUES BEFORE HANDLING")
print("=" * 70)

missing_before = X_production_clean.isnull().sum()
print(missing_before[missing_before > 0])

# Fill historical missing values using 0
# These occur because the first record of each machine
# has no previous production history.
historical_columns = [
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
]

X_production_clean[historical_columns] = (
    X_production_clean[historical_columns].fillna(0)
)

print("\nMISSING VALUES AFTER HANDLING")
print("=" * 70)

missing_after = X_production_clean.isnull().sum()
print(missing_after[missing_after > 0])

print("\nFINAL X SHAPE:")
print(X_production_clean.shape)

print("\nDATA TYPES:")
print(X_production_clean.dtypes)

MISSING VALUES BEFORE HANDLING
previous_units_produced     5
previous_rejection_rate     5
rolling_3_production_avg    5
rolling_3_rejection_avg     5
dtype: int64

MISSING VALUES AFTER HANDLING
Series([], dtype: int64)

FINAL X SHAPE:
(450, 32)

DATA TYPES:
machine_id                        int64
units_rejected                    int64
production_time_hours           float64
target_quantity                   int64
total_defects                     int64
defect_type_count                 int64
quality_defect_rate_percent     float64
total_downtime_hours            float64
downtime_event_count              int64
average_downtime_hours          float64
available_hours                 float64
operating_hours                 float64
utilization_percent             float64
sensor_value_mean               float64
sensor_value_min                float64
sensor_value_max                float64
sensor_value_std                float64
sensor_reading_count              int64
maintenance_event_cou

In [ ]:
# ============================================================

# TRAIN AND TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_production_clean,
    y_production,
    test_size=0.20,
    random_state=42
)

print("TRAIN / TEST SPLIT")
print("=" * 70)

print("\nX_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

print("\nTraining records:", len(X_train))
print("Testing records :", len(X_test))

TRAIN / TEST SPLIT

X_train shape: (360, 32)
X_test shape : (90, 32)

y_train shape: (360,)
y_test shape : (90,)

Training records: 360
Testing records : 90


In [ ]:

# DATA LEAKAGE CHECK


print("DATA LEAKAGE CHECK")
print("=" * 70)

# Check whether target exists inside features
print("\n1. Target column inside X:")
print("units_produced" in X_production_clean.columns)

# Check whether train and test have overlapping indexes
print("\n2. Train/Test index overlap:")
print(len(set(X_train.index) & set(X_test.index)))

# Check missing values
print("\n3. Missing values in X_train:")
print(X_train.isnull().sum().sum())

print("\n4. Missing values in X_test:")
print(X_test.isnull().sum().sum())

# Check shapes
print("\n5. Dataset shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

DATA LEAKAGE CHECK

1. Target column inside X:
False

2. Train/Test index overlap:
0

3. Missing values in X_train:
0

4. Missing values in X_test:
0

5. Dataset shapes:
X_train: (360, 32)
X_test : (90, 32)
y_train: (360,)
y_test : (90,)


In [ ]:

# FINAL VALIDATION & SAVE
# ============================================================

import os

# Create processed folder if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

print("FINAL ML DATASET VALIDATION")
print("=" * 70)

print("\nTraining Data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting Data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nMissing Values:")
print("X_train:", X_train.isnull().sum().sum())
print("X_test :", X_test.isnull().sum().sum())

# Save datasets
X_train.to_csv("../data/processed/X_train_production.csv", index=False)
X_test.to_csv("../data/processed/X_test_production.csv", index=False)

y_train.to_csv("../data/processed/y_train_production.csv", index=False)
y_test.to_csv("../data/processed/y_test_production.csv", index=False)

print("\nFiles saved successfully!")
print("Location: ../data/processed/")

FINAL ML DATASET VALIDATION

Training Data:
X_train: (360, 32)
y_train: (360,)

Testing Data:
X_test: (90, 32)
y_test: (90,)

Missing Values:
X_train: 0
X_test : 0

Files saved successfully!
Location: ../data/processed/


In [ ]:

# LOAD ML DATASETS
# ============================================================

import pandas as pd
import os

X_train = pd.read_csv("../data/processed/X_train_production.csv")
X_test = pd.read_csv("../data/processed/X_test_production.csv")

y_train = pd.read_csv("../data/processed/y_train_production.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test_production.csv").squeeze()

print("PRODUCTION ML DATASETS LOADED")
print("=" * 70)

print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nMissing values:")
print("X_train:", X_train.isnull().sum().sum())
print("X_test :", X_test.isnull().sum().sum())

print("\nTarget:")
print("units_produced")

print("\nTraining target statistics:")
print(y_train.describe())

PRODUCTION ML DATASETS LOADED

X_train: (360, 32)
X_test : (90, 32)
y_train: (360,)
y_test : (90,)

Missing values:
X_train: 0
X_test : 0

Target:
units_produced

Training target statistics:
count     360.000000
mean      908.686111
std        75.737185
min       670.000000
25%       863.750000
50%       920.000000
75%       959.250000
max      1134.000000
Name: units_produced, dtype: float64


In [ ]:

# BASELINE MODEL

from sklearn.linear_model import LinearRegression

# Create the model
model = LinearRegression()

# Train the model using our training data
model.fit(X_train, y_train)

# Make predictions on the test data
y_pred = model.predict(X_test)

print("Baseline model trained successfully!")
print("Number of predictions:", len(y_pred))

print("\nFirst 10 predictions:")
print(y_pred[:10])

Baseline model trained successfully!
Number of predictions: 90

First 10 predictions:
[940.18044146 926.42833272 874.51631373 928.58180235 905.74486972
 896.29530735 998.19171487 915.52977171 832.26755549 963.5269672 ]


In [ ]:

# CHECKING HOW GOOD OUR BASELINE MODEL IS
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Calculate the errors
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print("BASELINE MODEL RESULTS")
print("=" * 70)

print(f"\nMAE  : {mae:.2f} units")
print(f"RMSE : {rmse:.2f} units")
print(f"R²   : {r2:.4f}")

BASELINE MODEL RESULTS

MAE  : 23.92 units
RMSE : 30.56 units
R²   : 0.7719


In [ ]:
# ==================================
# TRYING RANDOM FOREST


from sklearn.ensemble import RandomForestRegressor

# Create the Random Forest model
random_forest = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

# Train the model
random_forest.fit(X_train, y_train)

# Predict production for the test data
rf_predictions = random_forest.predict(X_test)

print("Random Forest model trained successfully!")
print("Number of predictions:", len(rf_predictions))

print("\nFirst 10 predictions:")
print(rf_predictions[:10])

Random Forest model trained successfully!
Number of predictions: 90

First 10 predictions:
[933.205 925.07  898.925 948.63  891.67  912.215 990.455 921.625 797.235
 966.775]


In [ ]:

# EVALUATION OF RANDOM FOREST


# Calculate Random Forest errors
rf_mae = mean_absolute_error(y_test, rf_predictions)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))

rf_r2 = r2_score(y_test, rf_predictions)

print("RANDOM FOREST RESULTS")
print("=" * 70)

print(f"\nMAE  : {rf_mae:.2f} units")
print(f"RMSE : {rf_rmse:.2f} units")
print(f"R²   : {rf_r2:.4f}")

RANDOM FOREST RESULTS

MAE  : 31.28 units
RMSE : 39.73 units
R²   : 0.6143


In [ ]:

# TRYING GRADIENT BOOSTING
# ============================================================

from sklearn.ensemble import GradientBoostingRegressor

# Create the Gradient Boosting model
gradient_boosting = GradientBoostingRegressor
(
    n_estimators=100,
    random_state=42
)

# Train the model
gradient_boosting.fit(X_train, y_train)

# Predict production for the test data
gb_predictions = gradient_boosting.predict(X_test)

print("Gradient Boosting model trained successfully!")
print("Number of predictions:", len(gb_predictions))

print("\nFirst 10 predictions:")
print(gb_predictions[:10])

Gradient Boosting model trained successfully!
Number of predictions: 90

First 10 predictions:
[945.83330549 910.29675154 899.2943974  930.5894455  898.12943067
 898.45597506 993.93157048 917.84737917 797.32885573 968.37750159]


In [ ]:

# CHECK GRADIENT BOOSTING PERFORMANCE
# ============================================================

# Calculate how far the predictions are from the actual values
gb_mae = mean_absolute_error(y_test, gb_predictions)

# Calculate the RMSE
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_predictions))

# Calculate the R² score
gb_r2 = r2_score(y_test, gb_predictions)

print("GRADIENT BOOSTING RESULTS")
print("=" * 70)

print(f"\nMAE  : {gb_mae:.2f} units")
print(f"RMSE : {gb_rmse:.2f} units")
print(f"R²   : {gb_r2:.4f}")

GRADIENT BOOSTING RESULTS

MAE  : 30.59 units
RMSE : 37.58 units
R²   : 0.6549


In [ ]:
# VALIDATING THE BEST MODEL

from sklearn.model_selection import cross_val_score

# Check the model using 5 different splits
validation_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

print("R² scores from 5 validation runs:")
print(validation_scores)

print(f"\nAverage R²: {validation_scores.mean():.4f}")
print(f"Standard deviation: {validation_scores.std():.4f}")

R² scores from 5 validation runs:
[0.81873921 0.86650132 0.85997447 0.80226233 0.79297371]

Average R²: 0.8281
Standard deviation: 0.0299


In [ ]:
# Training the final Linear Regression model again

final_model = LinearRegression()

final_model.fit(X_train, y_train)

print("Final Linear Regression model trained successfully.")

Final Linear Regression model trained successfully.


In [ ]:
# Using the final model to predict production for the test data

final_predictions = final_model.predict(X_test)

print("Final production predictions generated successfully.")
print("Number of predictions:", len(final_predictions))

print("\nFirst 10 predictions:")
print(final_predictions[:10])

Final production predictions generated successfully.
Number of predictions: 90

First 10 predictions:
[940.18044146 926.42833272 874.51631373 928.58180235 905.74486972
 896.29530735 998.19171487 915.52977171 832.26755549 963.5269672 ]


In [ ]:
# Created a file with actual and predicted production values

production_predictions = pd.DataFrame({
    "actual_production": y_test.values,
    "predicted_production": final_predictions
})

# Calculate the difference between actual and predicted values
production_predictions["prediction_error"] = (
    production_predictions["actual_production"]
    - production_predictions["predicted_production"]
)

# Save the predictions
production_predictions.to_csv(
    "../data/processed/production_predictions.csv",
    index=False
)

print("Production predictions saved successfully.")
print("File: ../data/processed/production_predictions.csv")
print("Rows:", len(production_predictions))
print("Columns:", production_predictions.columns.tolist())

print("\nFirst 10 rows:")
print(production_predictions.head(10))

Production predictions saved successfully.
File: ../data/processed/production_predictions.csv
Rows: 90
Columns: ['actual_production', 'predicted_production', 'prediction_error']

First 10 rows:
   actual_production  predicted_production  prediction_error
0                931            940.180441         -9.180441
1                950            926.428333         23.571667
2                871            874.516314         -3.516314
3                899            928.581802        -29.581802
4                891            905.744870        -14.744870
5                900            896.295307          3.704693
6               1003            998.191715          4.808285
7                907            915.529772         -8.529772
8                845            832.267555         12.732445
9                948            963.526967        -15.526967


In [ ]:
# Load the feature engineered file

import pandas as pd

# Load the feature engineered file
quality_data = pd.read_csv("../data/processed/feature_engineered_data.csv")

print("Quality data loaded")
print("Shape:", quality_data.shape)

# Check the target column
print("\nTarget column: quality_defect_rate_percent")

# Check if the target has any missing values
print("Missing values:", quality_data["quality_defect_rate_percent"].isnull().sum())

Quality data loaded
Shape: (450, 42)

Target column: quality_defect_rate_percent
Missing values: 0


In [ ]:
# Set the column we want to predict
target_column = "quality_defect_rate_percent"

# Columns that are not useful for prediction
columns_to_remove = [
    "production_id",
    "target_id",
    "production_date",
    "target_date",
    "quality_defect_rate_percent"
]

# Create X by removing the columns above
X_quality = quality_data.drop(columns=columns_to_remove)

# Create y using the quality defect rate
y_quality = quality_data[target_column]

print("Quality prediction dataset created")

print("\nX shape:", X_quality.shape)
print("y shape:", y_quality.shape)

print("\nTarget:")
print(target_column)

print("\nFeatures:")
print(X_quality.columns.tolist())

Quality prediction dataset created

X shape: (450, 37)
y shape: (450,)

Target:
quality_defect_rate_percent

Features:
['machine_id', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']


In [ ]:
# Check if anything is missing
print("Missing values before handling:")
print(X_quality.isnull().sum()[X_quality.isnull().sum() > 0])

# Convert shift into numbers
X_quality["shift"] = X_quality["shift"].map({
    "Morning": 0,
    "Afternoon": 1,
    "Night": 2
})

# Check again after converting shift
print("\nMissing values after handling:")
print(X_quality.isnull().sum()[X_quality.isnull().sum() > 0])

print("\nFinal X shape:", X_quality.shape)
print("Target shape:", y_quality.shape)

Missing values before handling:
previous_units_produced     5
previous_rejection_rate     5
rolling_3_production_avg    5
rolling_3_rejection_avg     5
dtype: int64

Missing values after handling:
previous_units_produced     5
previous_rejection_rate     5
rolling_3_production_avg    5
rolling_3_rejection_avg     5
dtype: int64

Final X shape: (450, 37)
Target shape: (450,)


In [ ]:
# Fill the missing historical values with the median
history_columns = [
    "previous_units_produced",
    "previous_rejection_rate",
    "rolling_3_production_avg",
    "rolling_3_rejection_avg"
]

for column in history_columns:
    X_quality[column] = X_quality[column].fillna(X_quality[column].median())

# Check if any missing values are left
print("Missing values after handling:")
print(X_quality.isnull().sum().sum())

Missing values after handling:
0


In [ ]:
# Spliting training and testing data

from sklearn.model_selection import train_test_split

# Split the data into training and testing parts
X_train_quality, X_test_quality, y_train_quality, y_test_quality = train_test_split(
    X_quality,
    y_quality,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train_quality.shape)
print("Testing data:", X_test_quality.shape)

print("Training target:", y_train_quality.shape)
print("Testing target:", y_test_quality.shape)

Training data: (360, 37)
Testing data: (90, 37)
Training target: (360,)
Testing target: (90,)


In [ ]:
# Make sure the target is not inside our features
print("Target inside X:", target_column in X_quality.columns)

# Check for missing values
print("Missing values in training data:", X_train_quality.isnull().sum().sum())
print("Missing values in testing data:", X_test_quality.isnull().sum().sum())

# Check if training and testing rows overlap
print("Training/testing overlap:", len(
    set(X_train_quality.index) & set(X_test_quality.index)
))

Target inside X: False
Missing values in training data: 0
Missing values in testing data: 0
Training/testing overlap: 0


In [ ]:
# Train our first model


from sklearn.linear_model import LinearRegression

# Create the model
quality_model = LinearRegression()

# Train it using the training data
quality_model.fit(X_train_quality, y_train_quality)

# Predict the defect rate for the test data
quality_predictions = quality_model.predict(X_test_quality)

print("Quality prediction model trained successfully.")
print("Number of predictions:", len(quality_predictions))

print("\nFirst 10 predictions:")
print(quality_predictions[:10])


Quality prediction model trained successfully.
Number of predictions: 90

First 10 predictions:
[1.49101427 0.51753036 1.82275276 1.44424995 0.96451381 0.84885585
 1.46005733 2.07510161 0.35091246 0.20689628]


In [ ]:
# Evaluate the quality model


from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Check how far our predictions are from the actual values
quality_mae = mean_absolute_error(y_test_quality, quality_predictions)

quality_rmse = np.sqrt

(


    mean_squared_error(y_test_quality, quality_predictions)
)

quality_r2 = r2_score(y_test_quality, quality_predictions)

print("Quality prediction results")
print("=" * 50)

print(f"MAE  : {quality_mae:.4f}%")
print(f"RMSE : {quality_rmse:.4f}%")
print(f"R²   : {quality_r2:.4f}")

Quality prediction results
MAE  : 0.0186%
RMSE : 0.0299%
R²   : 0.9979


In [ ]:
# Removed columns that directly reveal the quality result
quality_leakage_columns = [
    "total_defects",
    "total_output"
]

X_quality_clean = X_quality.drop(columns=quality_leakage_columns)

print("Leakage columns removed")
print("New X shape:", X_quality_clean.shape)

Leakage columns removed
New X shape: (450, 35)


In [ ]:
# Spliting the clean data into training and testing parts
X_train_quality, X_test_quality, y_train_quality, y_test_quality = train_test_split(
    X_quality_clean,
    y_quality,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train_quality.shape)
print("Testing data:", X_test_quality.shape)

print("Training target:", y_train_quality.shape)
print("Testing target:", y_test_quality.shape)

Training data: (360, 35)
Testing data: (90, 35)
Training target: (360,)
Testing target: (90,)


In [ ]:
# Create a fresh model
quality_model = LinearRegression()

# Train the model
quality_model.fit(X_train_quality, y_train_quality)

# Predict the defect rate
quality_predictions = quality_model.predict(X_test_quality)

print("Quality model trained successfully")
print("Number of predictions:", len(quality_predictions))

print("\nFirst 10 predictions:")
print(quality_predictions[:10])

Quality model trained successfully
Number of predictions: 90

First 10 predictions:
[1.79803207 0.29326821 1.31467828 1.89426956 0.65344151 0.63210098
 1.18336358 1.39335174 0.44170308 0.21127675]


In [ ]:
# Check how accurate the new predictions are
quality_mae = mean_absolute_error(y_test_quality, quality_predictions)

quality_rmse = np.sqrt(
    mean_squared_error(y_test_quality, quality_predictions)
)

quality_r2 = r2_score(y_test_quality, quality_predictions)

print("Clean quality model results")
print("=" * 50)

print(f"MAE  : {quality_mae:.4f}%")
print(f"RMSE : {quality_rmse:.4f}%")
print(f"R²   : {quality_r2:.4f}")

Clean quality model results
MAE  : 0.2921%
RMSE : 0.3747%
R²   : 0.6762


In [ ]:
# Random Forest try

from sklearn.ensemble import RandomForestRegressor

# Create the Random Forest model
quality_rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

# Train the model
quality_rf.fit(X_train_quality, y_train_quality)

# Predict the defect rate
quality_rf_predictions = quality_rf.predict(X_test_quality)

print("Random Forest quality model trained successfully")
print("Number of predictions:", len(quality_rf_predictions))

print("\nFirst 10 predictions:")
print(quality_rf_predictions[:10])

Random Forest quality model trained successfully
Number of predictions: 90

First 10 predictions:
[1.84265 0.3851  1.13745 1.4385  0.7303  0.65795 1.03645 1.5874  0.38615
 0.24915]


In [ ]:
# Check how accurate the Random Forest predictions are
quality_rf_mae = mean_absolute_error(
    y_test_quality,
    quality_rf_predictions
)

quality_rf_rmse = np.sqrt(
    mean_squared_error(
        y_test_quality,
        quality_rf_predictions
    )
)

quality_rf_r2 = r2_score(
    y_test_quality,
    quality_rf_predictions
)

print("Random Forest quality results")
print("=" * 50)

print(f"MAE  : {quality_rf_mae:.4f}%")
print(f"RMSE : {quality_rf_rmse:.4f}%")
print(f"R²   : {quality_rf_r2:.4f}")

Random Forest quality results
MAE  : 0.2859%
RMSE : 0.3728%
R²   : 0.6794


In [ ]:
# Validate Random Forest
from sklearn.model_selection import cross_val_score

# Check how the model performs on different parts of the data
quality_validation_scores = cross_val_score(
    quality_rf,
    X_train_quality,
    y_train_quality,
    cv=5,
    scoring="r2"
)

print("Random Forest validation results")
print("=" * 50)

print("\nR² scores:")
print(quality_validation_scores)

print(f"\nAverage R²: {quality_validation_scores.mean():.4f}")
print(f"Standard deviation: {quality_validation_scores.std():.4f}")

Random Forest validation results

R² scores:
[0.72091587 0.69546918 0.60359698 0.60219383 0.7079197 ]

Average R²: 0.6660
Standard deviation: 0.0522


In [ ]:
# Create the final quality model
final_quality_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

# Train the final model
final_quality_model.fit(
    X_train_quality,
    y_train_quality
)

print("Final quality model trained successfully")

Final quality model trained successfully


In [ ]:
# Generate the final defect rate predictions
final_quality_predictions = final_quality_model.predict(X_test_quality)

print("Final quality predictions generated")
print("Number of predictions:", len(final_quality_predictions))

print("\nFirst 10 predictions:")
print(final_quality_predictions[:10])

Final quality predictions generated
Number of predictions: 90

First 10 predictions:
[1.84265 0.3851  1.13745 1.4385  0.7303  0.65795 1.03645 1.5874  0.38615
 0.24915]


In [ ]:
# Create a file with actual and predicted defect rates
quality_predictions_data = pd.DataFrame({
    "actual_defect_rate": y_test_quality.values,
    "predicted_defect_rate": final_quality_predictions
})

# Calculate the prediction error
quality_predictions_data["prediction_error"] = (
    quality_predictions_data["actual_defect_rate"]
    - quality_predictions_data["predicted_defect_rate"]
)

# Save the predictions
quality_predictions_data.to_csv(
    "../data/processed/quality_predictions.csv",
    index=False
)

print("Quality predictions saved successfully")
print("File: ../data/processed/quality_predictions.csv")
print("Rows:", len(quality_predictions_data))
print("Columns:", quality_predictions_data.columns.tolist())

print("\nFirst 10 rows:")
print(quality_predictions_data.head(10))

Quality predictions saved successfully
File: ../data/processed/quality_predictions.csv
Rows: 90
Columns: ['actual_defect_rate', 'predicted_defect_rate', 'prediction_error']

First 10 rows:
   actual_defect_rate  predicted_defect_rate  prediction_error
0                1.50                1.84265          -0.34265
1                0.52                0.38510           0.13490
2                1.85                1.13745           0.71255
3                1.45                1.43850           0.01150
4                0.98                0.73030           0.24970
5                0.87                0.65795           0.21205
6                1.43                1.03645           0.39355
7                2.08                1.58740           0.49260
8                0.35                0.38615          -0.03615
9                0.21                0.24915          -0.03915


In [ ]:
# Loaded the feature engineered data
failure_data = pd.read_csv("../data/processed/feature_engineered_data.csv")

print("Failure prediction data loaded")
print("Shape:", failure_data.shape)

# Check the maintenance related columns
print("\nMaintenance columns:")
print([
    "maintenance_event_count",
    "total_maintenance_downtime",
    "average_maintenance_downtime",
    "total_maintenance_cost",
    "completed_maintenance_count"
])

# Check some machine related data
print("\nSample data:")
print(failure_data[
    [
        "machine_id",
        "total_downtime_hours",
        "utilization_percent",
        "sensor_value_mean",
        "maintenance_event_count",
        "completed_maintenance_count"
    ]
].head(10))

Failure prediction data loaded
Shape: (450, 42)

Maintenance columns:
['maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count']

Sample data:
   machine_id  total_downtime_hours  utilization_percent  sensor_value_mean  \
0           1                  7.06                99.02              36.48   
1           1                  7.06                99.02              36.48   
2           1                  7.06                99.02              36.48   
3           1                  7.06                99.02              36.48   
4           1                  7.06                99.02              36.48   
5           1                  7.06                99.02              36.48   
6           1                  7.06                99.02              36.48   
7           1                  7.06                99.02              36.48   
8           1                  7.06                99.02  

In [ ]:
# Checking the machine related values

machine_data = failure_data[
    [
        "machine_id",
        "total_downtime_hours",
        "utilization_percent",
        "sensor_value_mean",
        "sensor_value_max",
        "sensor_value_std",
        "maintenance_event_count",
        "completed_maintenance_count"
    ]
].drop_duplicates()

print("Machine data:")
print(machine_data)

print("\nNumber of unique machines:", machine_data["machine_id"].nunique())

Machine data:
     machine_id  total_downtime_hours  utilization_percent  sensor_value_mean  \
0             1                  7.06                99.02              36.48   
90            2                 15.49                97.85              35.47   
180           3                 26.39                96.33              43.64   
270           4                 16.30                97.74              33.98   
360           5                  5.62                99.22              34.90   

     sensor_value_max  sensor_value_std  maintenance_event_count  \
0               76.56             33.65                        2   
90              75.40             32.86                        2   
180             88.87             38.73                        3   
270             75.49             31.59                        2   
360             75.02             32.34                        4   

     completed_maintenance_count  
0                              1  
90                  

In [ ]:
# Checking the columns available in the maintenance data

print("Columns in failure data:")
print(failure_data.columns.tolist())

Columns in failure data:
['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']


In [ ]:
# Checking the maintenance data columns
print(maintenance_df.columns.tolist())

# See a few maintenance records
print(maintenance_df.head())

NameError: name 'maintenance_df' is not defined

In [ ]:

print(failure_data.columns.tolist())

['production_id', 'machine_id', 'production_date', 'shift', 'units_produced', 'units_rejected', 'production_time_hours', 'target_id', 'target_date', 'target_quantity', 'target_achievement_percent', 'total_output', 'rejection_rate_percent', 'production_rate_per_hour', 'total_defects', 'defect_type_count', 'quality_defect_rate_percent', 'total_downtime_hours', 'downtime_event_count', 'average_downtime_hours', 'available_hours', 'operating_hours', 'utilization_percent', 'sensor_value_mean', 'sensor_value_min', 'sensor_value_max', 'sensor_value_std', 'sensor_reading_count', 'maintenance_event_count', 'total_maintenance_downtime', 'average_maintenance_downtime', 'total_maintenance_cost', 'completed_maintenance_count', 'production_day', 'production_month', 'production_day_of_week', 'is_weekend', 'shift_encoded', 'previous_units_produced', 'previous_rejection_rate', 'rolling_3_production_avg', 'rolling_3_rejection_avg']


In [ ]:
import os

# Check the files available in the data folders
print("Raw data files:")
print(os.listdir("../data/raw"))

print("\nProcessed data files:")
print(os.listdir("../data/processed"))

Raw data files:
[]

Processed data files:
['feature_engineered_data.csv', 'production_predictions.csv', 'quality_predictions.csv', 'X_test_production.csv', 'X_train_production.csv', 'y_test_production.csv', 'y_train_production.csv']


In [ ]:
# Check the values of important features for failure risk

risk_columns = [
    "total_downtime_hours",
    "downtime_event_count",
    "sensor_value_mean",
    "sensor_value_max",
    "sensor_value_std",
    "utilization_percent",
    "maintenance_event_count",
    "total_maintenance_downtime"
]

print(failure_data[risk_columns].describe())

       total_downtime_hours  downtime_event_count  sensor_value_mean  \
count            450.000000            450.000000         450.000000   
mean              14.172000              6.400000          36.894000   
std                7.481768              3.502465           3.473014   
min                5.620000              3.000000          33.980000   
25%                7.060000              4.000000          34.900000   
50%               15.490000              6.000000          35.470000   
75%               16.300000              6.000000          36.480000   
max               26.390000             13.000000          43.640000   

       sensor_value_max  sensor_value_std  utilization_percent  \
count        450.000000        450.000000           450.000000   
mean          78.268000         33.834000            98.032000   
std            5.331561          2.541365             1.040645   
min           75.020000         31.590000            96.330000   
25%           75.4000

In [ ]:
# Keep only one row for each machine

machine_risk_data = failure_data[
    [
        "machine_id",
        "total_downtime_hours",
        "downtime_event_count",
        "sensor_value_mean",
        "sensor_value_max",
        "sensor_value_std",
        "utilization_percent",
        "maintenance_event_count",
        "total_maintenance_downtime"
    ]
].drop_duplicates()

print("Machine level risk data")
print(machine_risk_data)

Machine level risk data
     machine_id  total_downtime_hours  downtime_event_count  \
0             1                  7.06                     4   
90            2                 15.49                     6   
180           3                 26.39                    13   
270           4                 16.30                     6   
360           5                  5.62                     3   

     sensor_value_mean  sensor_value_max  sensor_value_std  \
0                36.48             76.56             33.65   
90               35.47             75.40             32.86   
180              43.64             88.87             38.73   
270              33.98             75.49             31.59   
360              34.90             75.02             32.34   

     utilization_percent  maintenance_event_count  total_maintenance_downtime  
0                  99.02                        2                        6.74  
90                 97.85                        2               

In [ ]:
# Selectin the important values used for failure risk

risk_data = machine_risk_data[
    [
        "machine_id",
        "total_downtime_hours",
        "downtime_event_count",
        "sensor_value_max",
        "sensor_value_std",
        "maintenance_event_count"
    ]
].copy()

print("Risk data prepared")
print(risk_data)

Risk data prepared
     machine_id  total_downtime_hours  downtime_event_count  sensor_value_max  \
0             1                  7.06                     4             76.56   
90            2                 15.49                     6             75.40   
180           3                 26.39                    13             88.87   
270           4                 16.30                     6             75.49   
360           5                  5.62                     3             75.02   

     sensor_value_std  maintenance_event_count  
0               33.65                        2  
90              32.86                        2  
180             38.73                        3  
270             31.59                        2  
360             32.34                        4  


In [ ]:
# Import MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

# These are the columns used to calculate failure risk
risk_features = [
    "total_downtime_hours",
    "downtime_event_count",
    "sensor_value_max",
    "sensor_value_std",
    "maintenance_event_count"
]

# Create the scaler
scaler = MinMaxScaler()

# Convert the values into a scale from 0 to 1
risk_data[risk_features] = scaler.fit_transform(
    risk_data[risk_features]
)

print("Risk data after normalization")
print(risk_data)

ImportError: DLL load failed while importing _group_columns: An Application Control policy has blocked this file.

In [ ]:
# Columns used for failure risk calculation

risk_features = [
    "total_downtime_hours",
    "downtime_event_count",
    "sensor_value_max",
    "sensor_value_std",
    "maintenance_event_count"
]

# Normalize each column between 0 and 1
for column in risk_features:
    minimum = risk_data[column].min()
    maximum = risk_data[column].max()

    risk_data[column] = (
        (risk_data[column] - minimum)
        / (maximum - minimum)
    )

print("Risk data after normalization")
print(risk_data)

Risk data after normalization
     machine_id  total_downtime_hours  downtime_event_count  sensor_value_max  \
0             1              0.069331                   0.1          0.111191   
90            2              0.475205                   0.3          0.027437   
180           3              1.000000                   1.0          1.000000   
270           4              0.514203                   0.3          0.033935   
360           5              0.000000                   0.0          0.000000   

     sensor_value_std  maintenance_event_count  
0            0.288515                      0.0  
90           0.177871                      0.0  
180          1.000000                      0.5  
270          0.000000                      0.0  
360          0.105042                      1.0  


In [ ]:
# Calculate the average risk score for each machine

risk_data["failure_risk_score"] = risk_data[
    risk_features
].mean(axis=1)

print("Failure risk scores")
print(
    risk_data[
        [
            "machine_id",
            "failure_risk_score"
        ]
    ]
)

Failure risk scores
     machine_id  failure_risk_score
0             1            0.113808
90            2            0.196103
180           3            0.900000
270           4            0.169628
360           5            0.221008


In [ ]:
# Classification of each machine based on its failure risk score

def get_risk_level(score):
    if score < 0.20:
        return "Low Risk"
    elif score < 0.50:
        return "Medium Risk"
    else:
        return "High Risk"


# Add the risk level to the data
risk_data["failure_risk"] = risk_data[
    "failure_risk_score"
].apply(get_risk_level)

print("Failure risk classification")
print(
    risk_data[
        [
            "machine_id",
            "failure_risk_score",
            "failure_risk"
        ]
    ]
)

Failure risk classification
     machine_id  failure_risk_score failure_risk
0             1            0.113808     Low Risk
90            2            0.196103     Low Risk
180           3            0.900000    High Risk
270           4            0.169628     Low Risk
360           5            0.221008  Medium Risk


In [ ]:
# Finding the main reason for the machine risk

def get_risk_reason(row):
    reasons = []

    if row["total_downtime_hours"] > 20:
        reasons.append("High downtime")

    if row["downtime_event_count"] > 10:
        reasons.append("Frequent downtime")

    if row["sensor_value_max"] > 80:
        reasons.append("High sensor value")

    if row["sensor_value_std"] > 36:
        reasons.append("High sensor variation")

    if row["maintenance_event_count"] > 3:
        reasons.append("Frequent maintenance")

    if len(reasons) == 0:
        return "Normal operating conditions"

    return ", ".join(reasons)


risk_data["risk_reason"] = risk_data.apply(
    get_risk_reason,
    axis=1
)

print("Failure risk results")
print(
    risk_data[
        [
            "machine_id",
            "failure_risk_score",
            "failure_risk",
            "risk_reason"
        ]
    ]
)

Failure risk results
     machine_id  failure_risk_score failure_risk                  risk_reason
0             1            0.113808     Low Risk  Normal operating conditions
90            2            0.196103     Low Risk  Normal operating conditions
180           3            0.900000    High Risk  Normal operating conditions
270           4            0.169628     Low Risk  Normal operating conditions
360           5            0.221008  Medium Risk  Normal operating conditions


In [ ]:
# Adding the  reasons based on the original machine values

def get_risk_reason(row):
    reasons = []

    if row["total_downtime_hours"] > 20:
        reasons.append("High downtime")

    if row["downtime_event_count"] > 10:
        reasons.append("Frequent downtime")

    if row["sensor_value_max"] > 80:
        reasons.append("High sensor value")

    if row["sensor_value_std"] > 36:
        reasons.append("High sensor variation")

    if row["maintenance_event_count"] > 3:
        reasons.append("Frequent maintenance")

    if len(reasons) == 0:
        return "Normal operating conditions"

    return ", ".join(reasons)


# Get the reasons from the original values
machine_risk_data["risk_reason"] = machine_risk_data.apply(
    get_risk_reason,
    axis=1
)

# Add the reasons to our risk results
risk_data["risk_reason"] = machine_risk_data[
    "risk_reason"
].values

print("Final failure risk results")
print(
    risk_data[
        [
            "machine_id",
            "failure_risk_score",
            "failure_risk",
            "risk_reason"
        ]
    ]
)

Final failure risk results
     machine_id  failure_risk_score failure_risk  \
0             1            0.113808     Low Risk   
90            2            0.196103     Low Risk   
180           3            0.900000    High Risk   
270           4            0.169628     Low Risk   
360           5            0.221008  Medium Risk   

                                           risk_reason  
0                          Normal operating conditions  
90                         Normal operating conditions  
180  High downtime, Frequent downtime, High sensor ...  
270                        Normal operating conditions  
360                               Frequent maintenance  


In [ ]:
# Selecting the final columns we need

final_risk_results = risk_data[
    [
        "machine_id",
        "failure_risk_score",
        "failure_risk",
        "risk_reason"
    ]
].copy()

# Save the results
final_risk_results.to_csv(
    "../data/processed/failure_risk_predictions.csv",
    index=False
)

print("Failure risk results saved successfully")
print("File: ../data/processed/failure_risk_predictions.csv")
print("Rows:", len(final_risk_results))
print("Columns:", final_risk_results.columns.tolist())

print("\nFinal results:")
print(final_risk_results)

Failure risk results saved successfully
File: ../data/processed/failure_risk_predictions.csv
Rows: 5
Columns: ['machine_id', 'failure_risk_score', 'failure_risk', 'risk_reason']

Final results:
     machine_id  failure_risk_score failure_risk  \
0             1            0.113808     Low Risk   
90            2            0.196103     Low Risk   
180           3            0.900000    High Risk   
270           4            0.169628     Low Risk   
360           5            0.221008  Medium Risk   

                                           risk_reason  
0                          Normal operating conditions  
90                         Normal operating conditions  
180  High downtime, Frequent downtime, High sensor ...  
270                        Normal operating conditions  
360                               Frequent maintenance  


In [ ]:
# Checking all the final prediction files

production_results = pd.read_csv(
    "../data/processed/production_predictions.csv"
)

quality_results = pd.read_csv(
    "../data/processed/quality_predictions.csv"
)

failure_results = pd.read_csv(
    "../data/processed/failure_risk_predictions.csv"
)

print("Production results:", production_results.shape)
print("Quality results:", quality_results.shape)
print("Failure risk results:", failure_results.shape)

print("\nProduction columns:")
print(production_results.columns.tolist())

print("\nQuality columns:")
print(quality_results.columns.tolist())

print("\nFailure risk columns:")
print(failure_results.columns.tolist())

Production results: (90, 3)
Quality results: (90, 3)
Failure risk results: (5, 4)

Production columns:
['actual_production', 'predicted_production', 'prediction_error']

Quality columns:
['actual_defect_rate', 'predicted_defect_rate', 'prediction_error']

Failure risk columns:
['machine_id', 'failure_risk_score', 'failure_risk', 'risk_reason']


In [ ]:
# Checking the variables currently available in the notebook

print("Variables containing 'model':")

for name in globals():
    if "model" in name.lower():
        print(name)

Variables containing 'model':


In [ ]:
# Re-Loading linear regression model

X_train_production = pd.read_csv(
    "../data/processed/X_train_production.csv"
)

X_test_production = pd.read_csv(
    "../data/processed/X_test_production.csv"
)

y_train_production = pd.read_csv(
    "../data/processed/y_train_production.csv"
).squeeze()

y_test_production = pd.read_csv(
    "../data/processed/y_test_production.csv"
).squeeze()

print("Production data loaded")

print("X_train:", X_train_production.shape)
print("X_test:", X_test_production.shape)
print("y_train:", y_train_production.shape)
print("y_test:", y_test_production.shape)

Production data loaded
X_train: (360, 32)
X_test: (90, 32)
y_train: (360,)
y_test: (90,)


In [ ]:
# Train the final production model

from sklearn.linear_model import LinearRegression

final_production_model = LinearRegression()

final_production_model.fit(
    X_train_production,
    y_train_production
)

print("Final production model trained successfully")

Final production model trained successfully


In [ ]:
# Save the final production model

import joblib

joblib.dump(
    final_production_model,
    "../data/processed/production_model.pkl"
)

print("Production model saved successfully")
print("File: ../data/processed/production_model.pkl")

Production model saved successfully
File: ../data/processed/production_model.pkl


In [ ]:
# Loading the saved quality training and testing data

X_train_quality = pd.read_csv(
    "../data/processed/X_train_quality.csv"
)

X_test_quality = pd.read_csv(
    "../data/processed/X_test_quality.csv"
)

y_train_quality = pd.read_csv(
    "../data/processed/y_train_quality.csv"
).squeeze()

y_test_quality = pd.read_csv(
    "../data/processed/y_test_quality.csv"
).squeeze()

print("Quality data loaded")

print("X_train:", X_train_quality.shape)
print("X_test:", X_test_quality.shape)
print("y_train:", y_train_quality.shape)
print("y_test:", y_test_quality.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/X_train_quality.csv'

In [ ]:
# Checking the quality data variables available right now

print("Quality variables:")

for name in globals():
    if "quality" in name.lower() or "X_train" in name:
        print(name)

Quality variables:
quality
quality_results
X_train_production


In [ ]:
# Check for the quality dataset

print("Quality data shape:", quality.shape)

print("\nQuality columns:")
print(quality.columns.tolist())

Quality data shape: (450, 6)

Quality columns:
['quality_id', 'production_id', 'inspection_date', 'defect_type', 'defect_count', 'quality_status']


In [ ]:
# Loading the feature engineered data

data = pd.read_csv(
    "../data/processed/feature_engineered_data.csv"
)

print("Feature engineered data loaded")
print("Shape:", data.shape)

print("\nChecking quality target:")
print("quality_defect_rate_percent" in data.columns)

Feature engineered data loaded
Shape: (450, 42)

Checking quality target:
True


In [ ]:
# Seting the target column


quality_target = "quality_defect_rate_percent"

# Columns that should not be used for prediction
columns_to_remove = [
    "quality_defect_rate_percent",
    "production_id",
    "target_id",
    "target_date",
    "production_date"
]

# Create the input data and target
X_quality = data.drop(columns=columns_to_remove)
y_quality = data[quality_target]

print("Quality dataset prepared")
print("X shape:", X_quality.shape)
print("y shape:", y_quality.shape)

print("\nTarget:", quality_target)

Quality dataset prepared
X shape: (450, 37)
y shape: (450,)

Target: quality_defect_rate_percent


In [ ]:
# Removing the columns that can cause data leakage

leakage_columns = [
    "total_defects",
    "defect_type_count"
]

X_quality = X_quality.drop(columns=leakage_columns)

print("Leakage columns removed")
print("New X shape:", X_quality.shape)

Leakage columns removed
New X shape: (450, 35)


In [ ]:
# Split the quality data into training and testing data

from sklearn.model_selection import train_test_split

X_train_quality, X_test_quality, y_train_quality, y_test_quality = train_test_split(
    X_quality,
    y_quality,
    test_size=0.20,
    random_state=42
)

print("Quality data split completed")

print("Training data:", X_train_quality.shape)
print("Testing data:", X_test_quality.shape)
print("Training target:", y_train_quality.shape)
print("Testing target:", y_test_quality.shape)

Quality data split completed
Training data: (360, 35)
Testing data: (90, 35)
Training target: (360,)
Testing target: (90,)


In [ ]:


X_train_quality = X_train_quality.drop(columns=["shift"])
X_test_quality = X_test_quality.drop(columns=["shift"])

print("Text column removed")
print("Training data:", X_train_quality.shape)
print("Testing data:", X_test_quality.shape)

Text column removed
Training data: (360, 34)
Testing data: (90, 34)


In [129]:
# Train the final quality model

from sklearn.ensemble import RandomForestRegressor

final_quality_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

final_quality_model.fit(
    X_train_quality,
    y_train_quality
)

print("Final quality model trained successfully")

Final quality model trained successfully


In [130]:
# Saving

import joblib

joblib.dump(
    final_quality_model,
    "../data/processed/quality_model.pkl"
)

print("Quality model saved successfully")
print("File: ../data/processed/quality_model.pkl")

Quality model saved successfully
File: ../data/processed/quality_model.pkl


In [ ]:
# Checking the final files in the processed folder

import os

files = os.listdir("../data/processed")

print("Final processed files:")

for file in files:
    print(file)

Final processed files:
failure_risk_predictions.csv
feature_engineered_data.csv
production_model.pkl
production_predictions.csv
quality_model.pkl
quality_predictions.csv
X_test_production.csv
X_train_production.csv
y_test_production.csv
y_train_production.csv


In [132]:
# Checking the current notebook variables

print("Data + ML work completed")
print("Production model:", type(final_production_model).__name__)
print("Quality model:", type(final_quality_model).__name__)
print("Failure risk results:", len(final_risk_results))

Data + ML work completed
Production model: LinearRegression
Quality model: RandomForestRegressor
Failure risk results: 5


In [133]:
# ANOMOLY DETECTION
# Selecting the values we want to use for anomaly detection

anomaly_features = [
    "total_downtime_hours",
    "downtime_event_count",
    "utilization_percent",
    "sensor_value_mean",
    "sensor_value_max",
    "sensor_value_std",
    "maintenance_event_count"
]

# Create a separate dataframe for anomaly detection
anomaly_data = machine_risk_data[
    ["machine_id"] + anomaly_features
].copy()

print("Anomaly detection data prepared")
print("Shape:", anomaly_data.shape)

print("\nAnomaly features:")
print(anomaly_features)

print("\nSample data:")
print(anomaly_data)

Anomaly detection data prepared
Shape: (5, 8)

Anomaly features:
['total_downtime_hours', 'downtime_event_count', 'utilization_percent', 'sensor_value_mean', 'sensor_value_max', 'sensor_value_std', 'maintenance_event_count']

Sample data:
     machine_id  total_downtime_hours  downtime_event_count  \
0             1                  7.06                     4   
90            2                 15.49                     6   
180           3                 26.39                    13   
270           4                 16.30                     6   
360           5                  5.62                     3   

     utilization_percent  sensor_value_mean  sensor_value_max  \
0                  99.02              36.48             76.56   
90                 97.85              35.47             75.40   
180                96.33              43.64             88.87   
270                97.74              33.98             75.49   
360                99.22              34.90             7

In [134]:
# Training the anomaly detection model

from sklearn.ensemble import IsolationForest

anomaly_model = IsolationForest(
    contamination=0.20,
    random_state=42
)

anomaly_model.fit(
    anomaly_data[anomaly_features]
)

print("Anomaly detection model trained successfully")

Anomaly detection model trained successfully


In [135]:
# Prediction of  whether each machine is normal or abnormal

anomaly_data["anomaly_prediction"] = anomaly_model.predict(
    anomaly_data[anomaly_features]
)

# Get the anomaly score
anomaly_data["anomaly_score"] = anomaly_model.decision_function(
    anomaly_data[anomaly_features]
)

# Convert the prediction into an easy-to-read result
anomaly_data["anomaly_status"] = anomaly_data[
    "anomaly_prediction"
].map({
    1: "Normal",
    -1: "Anomaly"
})

print("Anomaly results generated")
print(
    anomaly_data[
        [
            "machine_id",
            "anomaly_score",
            "anomaly_status"
        ]
    ]
)

Anomaly results generated
     machine_id  anomaly_score anomaly_status
0             1       0.048116         Normal
90            2       0.148483         Normal
180           3      -0.125763        Anomaly
270           4       0.103202         Normal
360           5       0.031441         Normal


In [136]:
# simple reason for the anomaly

def get_anomaly_reason(row):
    reasons = []

    if row["total_downtime_hours"] > 20:
        reasons.append("High downtime")

    if row["downtime_event_count"] > 10:
        reasons.append("Frequent downtime")

    if row["sensor_value_max"] > 80:
        reasons.append("High sensor value")

    if row["sensor_value_std"] > 36:
        reasons.append("High sensor variation")

    if row["maintenance_event_count"] > 3:
        reasons.append("Frequent maintenance")

    if len(reasons) == 0:
        return "No unusual behaviour detected"

    return ", ".join(reasons)


anomaly_data["anomaly_reason"] = anomaly_data.apply(
    get_anomaly_reason,
    axis=1
)

print("Anomaly explanations added")
print(
    anomaly_data[
        [
            "machine_id",
            "anomaly_score",
            "anomaly_status",
            "anomaly_reason"
        ]
    ]
)

Anomaly explanations added
     machine_id  anomaly_score anomaly_status  \
0             1       0.048116         Normal   
90            2       0.148483         Normal   
180           3      -0.125763        Anomaly   
270           4       0.103202         Normal   
360           5       0.031441         Normal   

                                        anomaly_reason  
0                        No unusual behaviour detected  
90                       No unusual behaviour detected  
180  High downtime, Frequent downtime, High sensor ...  
270                      No unusual behaviour detected  
360                               Frequent maintenance  


In [137]:
# Selection of  the final anomaly results

final_anomaly_results = anomaly_data[
    [
        "machine_id",
        "anomaly_score",
        "anomaly_status",
        "anomaly_reason"
    ]
].copy()

# Save the results
final_anomaly_results.to_csv(
    "../data/processed/anomaly_detection_results.csv",
    index=False
)

print("Anomaly results saved successfully")
print("File: ../data/processed/anomaly_detection_results.csv")
print("Rows:", len(final_anomaly_results))
print("Columns:", final_anomaly_results.columns.tolist())

Anomaly results saved successfully
File: ../data/processed/anomaly_detection_results.csv
Rows: 5
Columns: ['machine_id', 'anomaly_score', 'anomaly_status', 'anomaly_reason']


In [139]:
# Checking the final ML output 

import os

processed_path = "../data/processed"

files_to_check = [
    "feature_engineered_data.csv",
    "production_predictions.csv",
    "quality_predictions.csv",
    "failure_risk_predictions.csv",
    "anomaly_detection_results.csv",
    "production_model.pkl",
    "quality_model.pkl"
]

print("Final Data + ML files")

for file in files_to_check:
    file_path = os.path.join(processed_path, file)

    if os.path.exists(file_path):
        print(file, "- PERFECTLY OK")
    else:
        print(file, "- Missing")

Final Data + ML files
feature_engineered_data.csv - PERFECTLY OK
production_predictions.csv - PERFECTLY OK
quality_predictions.csv - PERFECTLY OK
failure_risk_predictions.csv - PERFECTLY OK
anomaly_detection_results.csv - PERFECTLY OK
production_model.pkl - PERFECTLY OK
quality_model.pkl - PERFECTLY OK
